# 1편 micrograd: 신경망과 역전파를 손으로 만들기 (학생용)

앱의 단계 화면을 옆에 두고, 단계마다 빈 셀에 코드를 입력해 실행한 뒤 바로 아래 채점 셀을 실행하세요.

출처: 코드는 Andrej Karpathy의 micrograd, nn-zero-to-hero(MIT 라이선스)를 따라 만든 것입니다. https://github.com/karpathy/micrograd · https://github.com/karpathy/nn-zero-to-hero

- 그림 도구 graphviz: Colab에 기본으로 깔려 있는지는 미확인입니다. L01-S06 준비 코드 셀에서 `No module named 'graphviz'`가 나오면 새 셀에 `!pip install graphviz`를 직접 입력해 실행한 뒤 준비 코드 셀을 다시 실행하세요. 그래도 그림이 안 나오면 L01-S06 채점 셀의 안내대로 `check("L01-S06", seen=True)`를 씁니다.
- PyTorch(torch): L01-S13e에서 씁니다. Colab에 깔려 있는지는 이 노트북을 만들 때 확인하지 않았습니다(미확인). 없으면 L01-S13e 채점은 "건너뜀"으로 나옵니다.
- 채점 셀이 "○ 아직"이라고 하면 그 위 단계 셀을 아직 실행하지 않은 것입니다. 오류 대신 안내만 나옵니다.

## 준비 셀: 채점 도구

맨 처음 한 번 실행하세요. 채점 도구(z2h_check)를 셀 하나에 담은 것이라 길고, 내용은 고치지 않습니다. 실행하면 `z2h_check 준비 완료` 한 줄이 보입니다. Colab에서 런타임을 다시 시작했다면 이 셀부터 다시 실행합니다.

In [ ]:
# z2h_check 묶음 0.1.0: Z2H 노트북 채점 도구를 셀 하나에 담은 것입니다.
# 이 셀을 그대로 실행하면 check(...)와 finish("L01")를 쓸 수 있습니다. 내용은 고치지 마세요.
# 자동 생성 파일: 저장소에서 python -m z2h_check.bundle --write 로 다시 만듭니다 (원본: z2h_check/ 폴더).
# 원본 해시 sha256:bcae1572a84d80afc8ea3ab85c740e0b87f00193652864c186afecb390d46b24
# 출처 표기: 채점 기준의 강의 예제 숫자·데이터는 karpathy/nn-zero-to-hero, 비교 방식은 karpathy/micrograd (MIT).


def _z2h_install():
    import importlib.abc
    import importlib.util
    import sys

    sources = {
        'z2h_check': (
            '"""Z2H 노트북 채점 패키지.\n'
            '\n'
            '    from z2h_check import check, finish\n'
            '    check("L01-S11b", Value)     # ✓ 맞음: … / ✕ 다시: …\n'
            '    finish("L01")                # 채점표 + 결과코드 (+ 로컬 진도 서버로 보내기 시도)\n'
            '\n'
            '런타임 의존은 파이썬 표준 라이브러리뿐이다. torch는 있을 때만 쓴다.\n'
            '호출 규약은 z2h_check/README.md 를 본다.\n'
            '"""\n'
            '__version__ = "0.1.0"\n'
            '\n'
            'from .core import DEFAULT_SERVER_URL, check, finish, reset, results\n'
            'from .result_code import ResultCodeError, decode_result_code, encode_result_code\n'
            '\n'
            '__all__ = [\n'
            '    "check", "finish", "results", "reset",\n'
            '    "encode_result_code", "decode_result_code", "ResultCodeError",\n'
            '    "DEFAULT_SERVER_URL", "__version__",\n'
            ']\n'
        ),
        'z2h_check._resources': (
            '"""패키지 안 자료 파일(기준값 JSON) 읽기.\n'
            '\n'
            '묶음(_bundle.py)으로 실행될 때는 파일이 없으므로, 묶음이 _EMBEDDED 에 넣어 둔 글을 쓴다.\n'
            '"""\n'
            'import json\n'
            'import os\n'
            '\n'
            '_EMBEDDED = {}  # 묶음이 채운다: {"reference/l01.json": "..."}\n'
            '_CACHE = {}\n'
            '\n'
            '\n'
            'def read_text(relpath):\n'
            '    if relpath in _EMBEDDED:\n'
            '        return _EMBEDDED[relpath]\n'
            '    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), *relpath.split("/"))\n'
            '    with open(path, encoding="utf-8") as fh:\n'
            '        return fh.read()\n'
            '\n'
            '\n'
            'def load_json(relpath):\n'
            '    if relpath not in _CACHE:\n'
            '        _CACHE[relpath] = json.loads(read_text(relpath))\n'
            '    return _CACHE[relpath]\n'
        ),
        'z2h_check.core': (
            '"""check()와 finish(): 채점하고, 결과를 출력하고, 모듈 안에 기록한다.\n'
            '\n'
            '- check() 는 어떤 경우에도 예외를 밖으로 내보내지 않는다(Ctrl+C로 멈추는 것만 그대로 둔다).\n'
            '- 학생 코드가 틀리거나 비어 있으면 "✕ 다시", 채점 프로그램 자체가 고장 나면 "⚠ 채점기 오류"로 따로 보인다.\n'
            '- finish() 는 채점표와 결과코드를 늘 출력하고, 로컬 진도 서버(127.0.0.1)로 보내기를 시도한다.\n'
            '"""\n'
            'import contextlib\n'
            'import inspect\n'
            'import io\n'
            'import json\n'
            'import os\n'
            'import random\n'
            'import urllib.error\n'
            'import urllib.parse\n'
            'import urllib.request\n'
            'import warnings\n'
            '\n'
            'from . import graders\n'
            'from .graders import _util\n'
            'from .result_code import encode_result_code, now_kst_iso\n'
            '\n'
            'DEFAULT_SERVER_URL = "http://127.0.0.1:8787/api/grade"\n'
            'SEND_TIMEOUT_S = 2.0\n'
            '_LOCAL_HOSTS = {"127.0.0.1", "localhost", "::1"}\n'
            '_PKG_DIR = os.path.dirname(os.path.abspath(__file__))\n'
            '_GRADER_SEED = 20221116  # 채점 안에서 쓰는 무작위 값을 고정한다(학생의 무작위 상태는 채점 뒤 되돌린다)\n'
            '\n'
            'MARK = {"pass": "✓ 맞음", "fail": "✕ 다시", "skip": "· 건너뜀", "error": "⚠ 채점기 오류"}\n'
            '_ROW_MARK = {"pass": "✓", "fail": "✕", "skip": "·", "error": "⚠", "todo": "○"}\n'
            '\n'
            '_RESULTS = {}  # check_id -> {"check_id", "status", "passed", "message_ko", "at", "attempts"}\n'
            '\n'
            '\n'
            'def reset():\n'
            '    """기록을 모두 지운다."""\n'
            '    _RESULTS.clear()\n'
            '\n'
            '\n'
            'def results():\n'
            '    """지금까지의 채점 기록 사본. status: pass | fail | skip | error"""\n'
            '    return {cid: dict(rec) for cid, rec in _RESULTS.items()}\n'
            '\n'
            '\n'
            'def _emit(text):\n'
            '    print(text)\n'
            '\n'
            '\n'
            'def _say(text):\n'
            '    try:\n'
            '        _emit(text)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException:\n'
            '        pass\n'
            '\n'
            '\n'
            'def _short(x):\n'
            '    return repr(x) if isinstance(x, str) else _util.type_name(x)\n'
            '\n'
            '\n'
            '# ---- check ------------------------------------------------------------------\n'
            '\n'
            'def check(check_id, *args, **kwargs):\n'
            '    """check("L01-S11b", Value) -> True/False. 맞으면 "✓ 맞음: …", 다르면 "✕ 다시: …"를 출력한다."""\n'
            '    try:\n'
            '        return _check(check_id, args, kwargs)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:  # 마지막 안전망\n'
            '        _say(f"{MARK[\'error\']}: 채점 중 예상하지 못한 문제가 생겼습니다({type(exc).__name__}). 내 코드 탓이 아닐 수 있습니다.")\n'
            '        return False\n'
            '\n'
            '\n'
            'def _check(check_id, args, kwargs):\n'
            '    cid = graders.canonical_id(check_id)\n'
            '    if cid is None:\n'
            '        _say(f"{MARK[\'error\']}: 채점 번호 {_short(check_id)}를 모릅니다. 노트북의 채점 셀을 확인해 주세요. "\n'
            '             "(이 칸은 기록하지 않았습니다)")\n'
            '        return False\n'
            '    status, message = _run_grader(cid, args, kwargs)\n'
            '    prev = _RESULTS.get(cid)\n'
            '    _RESULTS[cid] = {\n'
            '        "check_id": cid,\n'
            '        "status": status,\n'
            '        "passed": status == "pass",\n'
            '        "message_ko": message,\n'
            '        "at": now_kst_iso(),\n'
            '        "attempts": (prev["attempts"] + 1) if prev else 1,\n'
            '    }\n'
            '    _say(f"{MARK[status]}: {message} [{cid}]")\n'
            '    return status == "pass"\n'
            '\n'
            '\n'
            '@contextlib.contextmanager\n'
            'def _quiet():\n'
            '    """채점하는 동안 학생 코드의 print·경고를 숨긴다."""\n'
            '    with contextlib.ExitStack() as stack:\n'
            '        stack.enter_context(contextlib.redirect_stdout(io.StringIO()))\n'
            '        stack.enter_context(contextlib.redirect_stderr(io.StringIO()))\n'
            '        stack.enter_context(warnings.catch_warnings())\n'
            '        warnings.simplefilter("ignore")\n'
            '        yield\n'
            '\n'
            '\n'
            'def _is_ours(filename):\n'
            '    return filename.startswith(_PKG_DIR + os.sep) or filename.startswith("<z2h_check")\n'
            '\n'
            '\n'
            'def _raised_in_package(exc):\n'
            '    tb = exc.__traceback__\n'
            '    last = None\n'
            '    while tb is not None:\n'
            '        last = tb\n'
            '        tb = tb.tb_next\n'
            '    return last is None or _is_ours(last.tb_frame.f_code.co_filename)\n'
            '\n'
            '\n'
            'def _run_grader(cid, args, kwargs):\n'
            '    fn = graders.get_grader(cid)\n'
            '    try:\n'
            '        bound = inspect.signature(fn).bind(*args, **kwargs)\n'
            '    except TypeError:\n'
            '        return "fail", f"채점 칸에 넘긴 값의 개수나 이름이 다릅니다. 이렇게 부르세요: {graders.usage_of(cid)}"\n'
            '    state = random.getstate()\n'
            '    try:\n'
            '        random.seed(_GRADER_SEED)\n'
            '        with _quiet():\n'
            '            message = fn(*bound.args, **bound.kwargs)\n'
            '        return "pass", str(message)\n'
            '    except _util.Fail as exc:\n'
            '        return "fail", exc.message_ko\n'
            '    except _util.Skip as exc:\n'
            '        return "skip", exc.message_ko\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        if _raised_in_package(exc):\n'
            '            return "error", (f"채점 프로그램 안에서 문제가 생겼습니다({_util.describe_exc(exc)}). "\n'
            '                             "내 코드 탓이 아닐 수 있으니 이 칸은 넘어가고 알려 주세요.")\n'
            '        return "fail", _util.student_error_message("내 코드를 실행", exc)\n'
            '    finally:\n'
            '        random.setstate(state)\n'
            '\n'
            '\n'
            '# ---- finish -----------------------------------------------------------------\n'
            '\n'
            'def finish(lecture_id="L01", send=True, server_url=None, timeout=SEND_TIMEOUT_S, at=None):\n'
            '    """이 편의 채점표를 출력하고 결과코드를 돌려준다. send=True면 로컬 진도 서버로도 보내 본다."""\n'
            '    try:\n'
            '        return _finish(lecture_id, send, server_url, timeout, at)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        _say(f"{MARK[\'error\']}: 채점표를 만들다 문제가 생겼습니다({type(exc).__name__}).")\n'
            '        return ""\n'
            '\n'
            '\n'
            'def _finish(lecture_id, send, server_url, timeout, at):\n'
            '    lid = graders.canonical_lecture(lecture_id)\n'
            '    if lid is None:\n'
            '        _say(f"{MARK[\'error\']}: 편 번호 {_short(lecture_id)}를 모릅니다. finish(\\"L01\\") 처럼 부르세요.")\n'
            '        return ""\n'
            '    items = graders.lecture_checks(lid)\n'
            '    at = at or now_kst_iso()\n'
            '    counts = {"pass": 0, "fail": 0, "skip": 0, "error": 0, "todo": 0}\n'
            '    passed, failed, rows = [], [], []\n'
            '    width = max(len(cid) for cid, _ in items)\n'
            '    for cid, title in items:\n'
            '        rec = _RESULTS.get(cid)\n'
            '        status = rec["status"] if rec else "todo"\n'
            '        counts[status] += 1\n'
            '        if status == "pass":\n'
            '            passed.append(cid)\n'
            '        elif status == "fail":\n'
            '            failed.append(cid)\n'
            '        tail = {"pass": "",\n'
            '                "fail": f": {rec[\'message_ko\']}" if rec else "",\n'
            '                "skip": f": 건너뜀 ({rec[\'message_ko\']})" if rec else "",\n'
            '                "error": ": 채점기 오류",\n'
            '                "todo": ": 아직 안 함"}[status]\n'
            '        rows.append(f"  {_ROW_MARK[status]} {cid.ljust(width)}  {title}{tail}")\n'
            '    head = (f"{lid} 채점표 (모두 {len(items)}칸): 통과 {counts[\'pass\']} · 다시 {counts[\'fail\']} · "\n'
            '            f"건너뜀 {counts[\'skip\']} · 아직 안 함 {counts[\'todo\']}")\n'
            '    if counts["error"]:\n'
            '        head += f" · 채점기 오류 {counts[\'error\']}"\n'
            '    _say("\\n".join([head] + rows))\n'
            '    code = encode_result_code(lid, passed, failed, at)\n'
            '    if send and os.environ.get("Z2H_NO_SEND", "") in ("", "0"):\n'
            '        _say(_send(lid, items, server_url, timeout, at))\n'
            '    _say("결과코드 (앱의 \'결과코드 붙여넣기\' 칸에 넣으세요):")\n'
            '    _say(code)\n'
            '    return code\n'
            '\n'
            '\n'
            'def _send(lid, items, server_url, timeout, at):\n'
            '    unreachable = "서버에 닿지 못했습니다. 아래 결과코드를 앱에 붙여 넣으세요. (Colab에서는 이것이 정상입니다)"\n'
            '    url = server_url or os.environ.get("Z2H_SERVER_URL") or DEFAULT_SERVER_URL\n'
            '    try:\n'
            '        parts = urllib.parse.urlsplit(url)\n'
            '        host = parts.hostname\n'
            '    except ValueError:\n'
            '        parts, host = None, None\n'
            '    if parts is None or parts.scheme != "http" or host not in _LOCAL_HOSTS:\n'
            '        return "진도 서버 주소는 http://127.0.0.1 만 씁니다. 보내지 않았습니다. 아래 결과코드를 앱에 붙여 넣으세요."\n'
            '    results_body = []\n'
            '    for cid, _title in items:\n'
            '        rec = _RESULTS.get(cid)\n'
            '        if rec and rec["status"] in ("pass", "fail"):\n'
            '            results_body.append({"check_id": cid, "passed": rec["status"] == "pass",\n'
            '                                 "message_ko": f"{MARK[rec[\'status\']]}: {rec[\'message_ko\']}"})\n'
            '    body = {"lecture_id": lid, "results": results_body, "at": at}\n'
            '    try:\n'
            '        status = _post_json(url, body, timeout)\n'
            '    except Exception:\n'
            '        return unreachable\n'
            '    if 200 <= status < 300:\n'
            '        return f"진도 서버에 채점 결과 {len(results_body)}칸을 보냈습니다. 앱을 새로 고치면 보입니다."\n'
            '    return f"서버가 결과를 받지 않았습니다(HTTP {status}). 아래 결과코드를 앱에 붙여 넣으세요."\n'
            '\n'
            '\n'
            'def _post_json(url, body, timeout):\n'
            '    """POST JSON. 프록시를 쓰지 않는다(127.0.0.1 전용). 돌려주는 값: HTTP 상태 번호."""\n'
            '    data = json.dumps(body, ensure_ascii=False).encode("utf-8")\n'
            '    parts = urllib.parse.urlsplit(url)\n'
            '    headers = {\n'
            '        "Content-Type": "application/json; charset=utf-8",\n'
            '        "Origin": f"{parts.scheme}://{parts.netloc}",\n'
            '        "User-Agent": "z2h-check/0.1",\n'
            '    }\n'
            '    req = urllib.request.Request(url, data=data, method="POST", headers=headers)\n'
            '    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))\n'
            '    try:\n'
            '        with opener.open(req, timeout=timeout) as resp:\n'
            '            return resp.status\n'
            '    except urllib.error.HTTPError as exc:\n'
            '        return exc.code\n'
        ),
        'z2h_check.graders': (
            '"""편별 채점 함수 모음. 지금은 1편(L01)만 있다."""\n'
            'from . import l01\n'
            '\n'
            '_LECTURES = {"L01": l01}\n'
            '\n'
            '\n'
            'def canonical_lecture(lecture_id):\n'
            '    """\'l01\', \' L01 \' -> \'L01\'. 모르는 편이면 None."""\n'
            '    if not isinstance(lecture_id, str):\n'
            '        return None\n'
            '    key = lecture_id.strip().upper()\n'
            '    return key if key in _LECTURES else None\n'
            '\n'
            '\n'
            'def lecture_checks(lecture_id):\n'
            '    """[(check_id, 제목)] 교사 자료 순서. 모르는 편이면 []."""\n'
            '    lid = canonical_lecture(lecture_id)\n'
            '    return list(_LECTURES[lid].CHECKS) if lid else []\n'
            '\n'
            '\n'
            'def canonical_id(check_id):\n'
            '    """대소문자를 봐주고 정식 check_id 를 돌려준다 (\'l01-s11b\' -> \'L01-S11b\'). 모르면 None."""\n'
            '    if not isinstance(check_id, str):\n'
            '        return None\n'
            '    key = check_id.strip().lower()\n'
            '    for mod in _LECTURES.values():\n'
            '        for cid, _title in mod.CHECKS:\n'
            '            if cid.lower() == key:\n'
            '                return cid\n'
            '    return None\n'
            '\n'
            '\n'
            'def _module_of(check_id):\n'
            '    return _LECTURES[check_id.split("-", 1)[0]]\n'
            '\n'
            '\n'
            'def get_grader(check_id):\n'
            '    return _module_of(check_id).GRADERS[check_id]\n'
            '\n'
            '\n'
            'def usage_of(check_id):\n'
            '    return _module_of(check_id).USAGE[check_id]\n'
            '\n'
            '\n'
            'def title_of(check_id):\n'
            '    return dict(_module_of(check_id).CHECKS)[check_id]\n'
        ),
        'z2h_check.graders._util': (
            '"""채점 함수들이 같이 쓰는 도구.\n'
            '\n'
            '원칙\n'
            '- 학생 코드(넘긴 클래스·함수·값)는 늘 student()로 감싸 부른다. 거기서 난 오류는 Fail("✕ 다시")이 된다.\n'
            '- 학생 값을 숫자로 바꿀 때는 as_number(), 속성은 get_attr()/try_attr()를 쓴다. 이 도구들은 어떤 입력에도\n'
            '  Fail 말고 다른 예외를 내지 않는다. 그래서 이 밖에서 난 예외는 채점기 자체의 오류로 본다.\n'
            '"""\n'
            'import math\n'
            'import os\n'
            'import re\n'
            '\n'
            '\n'
            'class Fail(Exception):\n'
            '    """학생 결과가 기준과 다름 ("✕ 다시"). message_ko 는 무엇이 다른지."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'class Skip(Exception):\n'
            '    """채점할 수 없어 건너뜀 (예: torch 없음). 통과도 실패도 아니다."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'TORCH_SKIP_NOTE = "PyTorch 비교는 torch가 없어 건너뜀"\n'
            'MISSING = object()\n'
            '\n'
            '\n'
            '# ---- 오류 설명 -------------------------------------------------------------\n'
            '\n'
            'def _safe_str(exc):\n'
            '    try:\n'
            '        text = str(exc)\n'
            '    except Exception:\n'
            '        text = "(오류 내용을 읽지 못함)"\n'
            '    text = " ".join(text.split())\n'
            '    return text if len(text) <= 160 else text[:157] + "..."\n'
            '\n'
            '\n'
            'def describe_exc(exc):\n'
            '    name = type(exc).__name__\n'
            '    text = _safe_str(exc)\n'
            '    return f"{name}: {text}" if text else name\n'
            '\n'
            '\n'
            '_WRAP_HINT = "숫자와 섞어 계산하려면 숫자를 Value로 감싸세요: other = other if isinstance(other, Value) else Value(other)"\n'
            '\n'
            '\n'
            'def hint_for(exc):\n'
            '    """자주 막히는 오류에 붙이는 한 줄 도움말 (research/01 5절, pedagogy/01 7절)."""\n'
            '    text = _safe_str(exc)\n'
            '    if isinstance(exc, NameError) and "___" in text:\n'
            '        return "빈칸(___)이 아직 남아 있습니다. 빈칸을 채우고 그 셀부터 다시 실행하세요."\n'
            '    if isinstance(exc, NameError):\n'
            '        return "아직 정의하지 않은 이름이 있습니다. 앞 셀들을 위에서부터 다시 실행했나요?"\n'
            '    if isinstance(exc, NotImplementedError):\n'
            '        return "아직 채우지 않은 부분이 있습니다."\n'
            '    if isinstance(exc, RecursionError):\n'
            '        return "같은 값을 끝없이 따라갑니다. 위상 정렬에서 visited로 이미 본 값을 건너뛰나요?"\n'
            '    if isinstance(exc, ZeroDivisionError):\n'
            '        return "0으로 나눴습니다."\n'
            '    if isinstance(exc, OverflowError):\n'
            '        return "숫자가 너무 커졌습니다."\n'
            '    if isinstance(exc, TypeError) and "NoneType" in text:\n'
            '        return "빈칸(___)을 아직 채우지 않아 None이 남아 있을 수 있습니다. 빈칸을 채우고 셀을 다시 실행하세요."\n'
            '    if isinstance(exc, TypeError) and "unsupported operand" in text:\n'
            '        m = re.search(r"for (\\S+): \'(\\w+)\' and \'(\\w+)\'", text)\n'
            '        if m:\n'
            '            op, left, right = m.groups()\n'
            '            if left in ("int", "float") and right not in ("int", "float"):\n'
            '                name = {"+": "__radd__", "*": "__rmul__", "-": "__rsub__", "/": "__rtruediv__"}.get(op)\n'
            '                if op == "+":\n'
            '                    return "숫자 + Value 를 하려면 __radd__가 필요합니다(sum()은 0부터 더합니다). 또는 sum(..., Value(0.0))처럼 시작값을 주세요."\n'
            '                if name:\n'
            '                    return f"숫자 {op} Value 를 하려면 {name}가 필요합니다."\n'
            '            if right in ("int", "float"):\n'
            '                return _WRAP_HINT\n'
            '    if isinstance(exc, TypeError) and ("not callable" in text or "takes" in text or "argument" in text):\n'
            '        return "넘긴 것이 함수·클래스 모양과 다릅니다. 셀의 정의를 다시 실행했나요?"\n'
            '    if isinstance(exc, AttributeError):\n'
            '        if re.search(r"\'(int|float)\' object has no attribute \'(data|grad|_prev)\'", text):\n'
            '            return _WRAP_HINT\n'
            '        m = re.search(r"has no attribute \'(\\w+)\'", text)\n'
            '        if m:\n'
            '            return f"{m.group(1)} 이(가) 아직 없습니다. 그 단계의 코드를 넣고 클래스 셀을 다시 실행했나요?"\n'
            '    return ""\n'
            '\n'
            '\n'
            'def student_error_message(what, exc):\n'
            '    hint = hint_for(exc)\n'
            '    msg = f"{what} 중 오류가 났습니다: {describe_exc(exc)}."\n'
            '    return f"{msg} {hint}" if hint else msg\n'
            '\n'
            '\n'
            'def student(what, fn, *args, **kwargs):\n'
            '    """학생 코드를 부른다. 오류가 나면 Fail(무엇을 하다 무슨 오류가 났는지)로 바꾼다."""\n'
            '    try:\n'
            '        return fn(*args, **kwargs)\n'
            '    except (KeyboardInterrupt, Fail, Skip):\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        raise Fail(student_error_message(what, exc)) from None\n'
            '\n'
            '\n'
            '# ---- 값 읽기 ---------------------------------------------------------------\n'
            '\n'
            'def type_name(x):\n'
            '    try:\n'
            '        return type(x).__name__\n'
            '    except Exception:\n'
            '        return "알 수 없는 것"\n'
            '\n'
            '\n'
            'def fmt(x):\n'
            '    """화면용 숫자: 2.0, -3.0, 14.0003, 0.881374"""\n'
            '    if isinstance(x, bool):\n'
            '        return str(x)\n'
            '    if isinstance(x, int):\n'
            '        return str(x)\n'
            '    if isinstance(x, float):\n'
            '        if math.isfinite(x) and x == int(x) and abs(x) < 1e15:\n'
            '            return f"{x:.1f}"\n'
            '        return f"{x:.6g}"\n'
            '    return str(x)\n'
            '\n'
            '\n'
            'def short(x):\n'
            '    """학생 값을 안전하게 짧게 보여 준다 (글자는 따옴표, 나머지는 종류 이름)."""\n'
            '    if isinstance(x, str):\n'
            '        return repr(x[:30])\n'
            '    if isinstance(x, (int, float)) and not isinstance(x, bool):\n'
            '        return fmt(x)\n'
            '    return f"{type_name(x)} 값"\n'
            '\n'
            '\n'
            'def is_blank(x):\n'
            '    return x is None or x is Ellipsis\n'
            '\n'
            '\n'
            'def need(x, label):\n'
            '    if is_blank(x):\n'
            '        raise Fail(f"{label} 칸이 아직 비어 있습니다. 값을 넣고 셀을 다시 실행하세요.")\n'
            '    return x\n'
            '\n'
            '\n'
            'def need_callable(x, label, usage):\n'
            '    need(x, label)\n'
            '    if not callable(x):\n'
            '        raise Fail(f"{label} 자리에 {short(x)}이(가) 왔습니다. {usage} 처럼 클래스(함수) 이름을 넘기세요.")\n'
            '    return x\n'
            '\n'
            '\n'
            'def try_attr(obj, name):\n'
            '    """속성을 읽되, 없거나 오류가 나면 MISSING."""\n'
            '    try:\n'
            '        return getattr(obj, name)\n'
            '    except Exception:\n'
            '        return MISSING\n'
            '\n'
            '\n'
            'def has_attr(obj, name):\n'
            '    return try_attr(obj, name) is not MISSING\n'
            '\n'
            '\n'
            'def get_attr(obj, name, owner, hint=""):\n'
            '    try:\n'
            '        return getattr(obj, name)\n'
            '    except AttributeError:\n'
            '        extra = f" {hint}" if hint else ""\n'
            '        raise Fail(f"{owner}에서 \'{name}\'을(를) 찾지 못했습니다.{extra}") from None\n'
            '    except (KeyboardInterrupt, Fail, Skip):\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        raise Fail(student_error_message(f"{owner}.{name} 읽기", exc)) from None\n'
            '\n'
            '\n'
            'def as_number(x, label, _depth=0):\n'
            '    """학생 값을 float로. Value처럼 .data가 있으면 .data, 텐서·numpy처럼 .item()이 있으면 그 값."""\n'
            '    need(x, label)\n'
            '    if isinstance(x, bool):\n'
            '        raise Fail(f"{label}에는 숫자가 와야 하는데 {x}가 왔습니다.")\n'
            '    if isinstance(x, (int, float)):\n'
            '        v = float(x)\n'
            '    elif isinstance(x, (str, bytes)):\n'
            '        raise Fail(f"{label}에는 숫자가 와야 하는데 글자 {short(x)}이(가) 왔습니다.")\n'
            '    elif _depth >= 3:\n'
            '        raise Fail(f"{label}을(를) 숫자로 바꾸지 못했습니다({type_name(x)}).")\n'
            '    else:\n'
            '        item = try_attr(x, "item")\n'
            '        if item is not MISSING and callable(item):\n'
            '            return as_number(student(f"{label}을(를) 숫자로 바꾸기", item), label, _depth + 1)\n'
            '        data = try_attr(x, "data")\n'
            '        if data is not MISSING:\n'
            '            return as_number(data, label, _depth + 1)\n'
            '        v = student(f"{label}을(를) 숫자로 바꾸기", float, x)\n'
            '        if not isinstance(v, float):\n'
            '            raise Fail(f"{label}을(를) 숫자로 바꾸지 못했습니다({type_name(x)}).")\n'
            '    if not math.isfinite(v):\n'
            '        raise Fail(f"{label}: 내 값 {v}. 계산 중 숫자가 넘쳤거나 0으로 나눴습니다.")\n'
            '    return v\n'
            '\n'
            '\n'
            'def number_list(x, label, min_len=2):\n'
            '    need(x, label)\n'
            '    if isinstance(x, (str, bytes)):\n'
            '        raise Fail(f"{label}에는 숫자 목록이 와야 하는데 글자가 왔습니다.")\n'
            '    items = student(f"{label} 읽기", list, x)\n'
            '    if len(items) < min_len:\n'
            '        raise Fail(f"{label}에 숫자가 {len(items)}개뿐입니다. 반복마다 loss.data 를 목록에 모아 {min_len}개 이상 넘기세요.")\n'
            '    return [as_number(v, f"{label}[{i}]") for i, v in enumerate(items)]\n'
            '\n'
            '\n'
            'def close(got, want, tol):\n'
            '    return abs(got - want) <= tol * max(1.0, abs(want))\n'
            '\n'
            '\n'
            '# ---- Value 다루기 -----------------------------------------------------------\n'
            '\n'
            'def is_value_like(x):\n'
            '    return not is_blank(x) and has_attr(x, "data") and has_attr(x, "grad")\n'
            '\n'
            '\n'
            'def require_value_like(x, label):\n'
            '    need(x, label)\n'
            '    if not is_value_like(x):\n'
            '        raise Fail(f"{label}이(가) Value가 아닙니다(받은 것: {type_name(x)}). data와 grad가 있는 Value여야 합니다.")\n'
            '    return x\n'
            '\n'
            '\n'
            'def make_value(Value, x):\n'
            '    out = student(f"Value({fmt(x)}) 만들기", Value, x)\n'
            '    require_value_like(out, f"Value({fmt(x)})")\n'
            '    return out\n'
            '\n'
            '\n'
            'def data_of(v, label):\n'
            '    return as_number(get_attr(v, "data", label), f"{label}.data")\n'
            '\n'
            '\n'
            'def grad_of(v, label):\n'
            '    return as_number(get_attr(v, "grad", label), f"{label}.grad")\n'
            '\n'
            '\n'
            'def set_grad(v, g, label):\n'
            '    student(f"{label}.grad = {fmt(g)} 넣기", setattr, v, "grad", g)\n'
            '\n'
            '\n'
            'def prev_of(v, label):\n'
            '    p = get_attr(v, "_prev", label, hint="__init__에 self._prev = set(_children) 를 넣으세요.")\n'
            '    if isinstance(p, (str, bytes)) or is_blank(p):\n'
            '        raise Fail(f"{label}._prev 에는 재료 값들의 모음(set)이 와야 합니다.")\n'
            '    return student(f"{label}._prev 읽기", list, p)\n'
            '\n'
            '\n'
            'def same_items(items, expected):\n'
            '    """두 목록이 같은 객체들(같은 id)인지. _prev 비교용."""\n'
            '    return sorted(id(x) for x in items) == sorted(id(x) for x in expected)\n'
            '\n'
            '\n'
            'def walk(root, label="결과 값"):\n'
            '    """_prev를 따라 root에 이어진 값을 재료부터 결과 순으로 모은다 (학생 backward와 무관한 채점기 자체 정렬)."""\n'
            '    order, seen, stack = [], set(), [(root, False)]\n'
            '    while stack:\n'
            '        node, done = stack.pop()\n'
            '        if done:\n'
            '            order.append(node)\n'
            '            continue\n'
            '        if id(node) in seen:\n'
            '            continue\n'
            '        seen.add(id(node))\n'
            '        if len(seen) > 100000:\n'
            '            raise Fail(f"{label}에 이어진 값이 너무 많습니다(10만 개 넘음).")\n'
            '        stack.append((node, True))\n'
            '        for child in prev_of(node, "값"):\n'
            '            if id(child) not in seen:\n'
            '                stack.append((child, False))\n'
            '    return order\n'
            '\n'
            '\n'
            'def drive_backward(root):\n'
            '    """학생 backward() 대신 채점기가 직접 거꾸로 돈다: 맨 끝 grad = 1.0, 결과부터 재료 순으로 _backward()."""\n'
            '    order = walk(root)\n'
            '    set_grad(root, 1.0, "맨 끝 값")\n'
            '    hint = "__init__에 self._backward = lambda: None 을 넣고, 연산마다 out._backward = _backward 로 넣으세요."\n'
            '    for node in reversed(order):\n'
            '        bw = get_attr(node, "_backward", "Value", hint=hint)\n'
            '        student("_backward() 실행", bw)\n'
            '\n'
            '\n'
            '# ---- PyTorch 비교 ----------------------------------------------------------\n'
            '\n'
            '_TORCH = {"tried": False, "mod": None}\n'
            '\n'
            '\n'
            'def get_torch():\n'
            '    """torch가 있으면 모듈, 없으면 None. 환경변수 Z2H_NO_TORCH=1 이면 없는 것으로 친다."""\n'
            '    if os.environ.get("Z2H_NO_TORCH", "") not in ("", "0"):\n'
            '        return None\n'
            '    if not _TORCH["tried"]:\n'
            '        _TORCH["tried"] = True\n'
            '        try:\n'
            '            import torch\n'
            '            _TORCH["mod"] = torch\n'
            '        except Exception:\n'
            '            _TORCH["mod"] = None\n'
            '    return _TORCH["mod"]\n'
            '\n'
            '\n'
            'class Expr:\n'
            '    """학생 Value와 PyTorch 텐서 둘 다에 쓸 수 있는 식 하나 (연산자, .tanh(), .exp()만 씀)."""\n'
            '\n'
            '    def __init__(self, desc, fn, values):\n'
            '        self.desc = desc\n'
            '        self.fn = fn\n'
            '        self.values = tuple(values)\n'
            '        self.names = fn.__code__.co_varnames[:fn.__code__.co_argcount]\n'
            '\n'
            '\n'
            'def run_value_expr(Value, expr, use_student_backward):\n'
            '    leaves = [make_value(Value, v) for v in expr.values]\n'
            '    out = student(f"식 {expr.desc} 계산", expr.fn, *leaves)\n'
            '    require_value_like(out, f"식 {expr.desc}의 결과")\n'
            '    if use_student_backward:\n'
            '        bw = get_attr(out, "backward", "Value", hint="Value에 backward() 메서드를 넣으세요.")\n'
            '        student(f"식 {expr.desc}에서 backward() 실행", bw)\n'
            '    else:\n'
            '        drive_backward(out)\n'
            '    return data_of(out, "결과"), [grad_of(leaf, name) for leaf, name in zip(leaves, expr.names)]\n'
            '\n'
            '\n'
            'def run_torch_expr(torch, expr):\n'
            '    leaves = [torch.tensor(float(v), dtype=torch.float64, requires_grad=True) for v in expr.values]\n'
            '    out = expr.fn(*leaves)\n'
            '    out.backward()\n'
            '    grads = [float(leaf.grad.item()) if leaf.grad is not None else 0.0 for leaf in leaves]\n'
            '    return float(out.item()), grads\n'
            '\n'
            '\n'
            'def compare_with_torch(Value, exprs, use_student_backward, tol, hint=""):\n'
            '    """식마다 학생 값·기울기를 PyTorch(float64)와 비교. torch가 없으면 False(건너뜀), 다 같으면 True."""\n'
            '    torch = get_torch()\n'
            '    if torch is None:\n'
            '        return False\n'
            '    extra = f" {hint}" if hint else ""\n'
            '    for expr in exprs:\n'
            '        got_out, got_grads = run_value_expr(Value, expr, use_student_backward)\n'
            '        want_out, want_grads = run_torch_expr(torch, expr)\n'
            '        if not close(got_out, want_out, tol):\n'
            '            raise Fail(f"식 {expr.desc} 의 값: 내 값 {fmt(got_out)}, PyTorch {fmt(want_out)}.{extra}")\n'
            '        for name, got, want in zip(expr.names, got_grads, want_grads):\n'
            '            if not close(got, want, tol):\n'
            '                raise Fail(f"식 {expr.desc} 에서 {name}.grad: 내 값 {fmt(got)}, PyTorch {fmt(want)}.{extra}")\n'
            '    return True\n'
            '\n'
            '\n'
            'def torch_note(done, count):\n'
            '    return f" PyTorch와도 식 {count}개에서 같습니다." if done else f" ({TORCH_SKIP_NOTE})"\n'
        ),
        'z2h_check.graders.l01': (
            '# 출처: 강의 예제 숫자와 학습 데이터는 karpathy/nn-zero-to-hero (lectures/micrograd), PyTorch 기울기 비교 방식은 karpathy/micrograd (test/test_engine.py), 둘 다 MIT License. 식과 채점 코드는 새로 씀.\n'
            '"""1편 micrograd 채점 함수: 교사 정본(inputs/pedagogy/content/L01.json)의 단계 전부 + 재구성 L01-R1 ~ L01-R3.\n'
            '\n'
            '- 단계 id·순서·내용은 교사 정본을 따른다. 개수는 아래 @grader 등록 목록(CHECKS)에서 센다(코드에 박지 않음).\n'
            '- 호출 규약(넘기는 인자)은 각 함수의 usage와 z2h_check/README.md 표가 정본이다.\n'
            '- 맞으면 짧은 문장을 돌려주고, 다르면 Fail("무엇이 다른지")을 던진다. 학생 코드는 U.student()로만 부른다.\n'
            '- 기울기는 학생 Value로 계산해 같은 식의 PyTorch 기울기, 또는 손으로 계산한 값(reference/l01.json)과 비교한다.\n'
            '- 학생 backward()를 시험하는 칸(S11b, R2)만 학생 backward()를 쓰고, 나머지는 채점기가 직접 거꾸로 돈다\n'
            '  (U.drive_backward). 그래서 backward()가 틀려도 S12·S13 묶음·R1은 그 단계 내용만 따로 채점된다.\n'
            '"""\n'
            'import math\n'
            '\n'
            'from . import _util as U\n'
            'from .._resources import load_json\n'
            '\n'
            'CHECKS = []   # [(check_id, title)] 교사 정본 순서(등록 순서 그대로)\n'
            'GRADERS = {}  # check_id -> 채점 함수\n'
            'USAGE = {}    # check_id -> 호출 예\n'
            '\n'
            '\n'
            'def grader(check_id, title, usage):\n'
            '    def deco(fn):\n'
            '        CHECKS.append((check_id, title))\n'
            '        GRADERS[check_id] = fn\n'
            '        USAGE[check_id] = usage\n'
            '        return fn\n'
            '    return deco\n'
            '\n'
            '\n'
            'def ref():\n'
            '    return load_json("reference/l01.json")\n'
            '\n'
            '\n'
            'def tol():\n'
            '    return ref()["tolerance"]["grad"]\n'
            '\n'
            '\n'
            'F = U.fmt\n'
            '\n'
            '\n'
            '# ---- 여러 칸이 같이 쓰는 식 --------------------------------------------------\n'
            '\n'
            'def _tanh_by_exp(n):\n'
            '    e = (2 * n).exp()\n'
            '    return (e - 1) / (e + 1)\n'
            '\n'
            '\n'
            '# S11b: 덧셈·곱셈·tanh만, 값마다 한 번씩만 쓰는 식 (+= 버그와 섞이지 않게)\n'
            'TREE_EXPRS = [\n'
            '    U.Expr("(a*b + c).tanh() * d", lambda a, b, c, d: (a * b + c).tanh() * d, (0.5, -1.5, 0.3, 2.0)),\n'
            '    U.Expr("(a*b + c*d).tanh() + e", lambda a, b, c, d, e: (a * b + c * d).tanh() + e, (1.2, -0.7, 0.4, 0.9, -2.0)),\n'
            '    U.Expr("((a*b + c).tanh()*d + e).tanh() * f",\n'
            '           lambda a, b, c, d, e, f: ((a * b + c).tanh() * d + e).tanh() * f, (0.8, -1.1, 0.2, 1.7, -0.3, 2.5)),\n'
            ']\n'
            '# R1, R2: 같은 값을 여러 번 쓰는 식 (완성본은 += 로 쌓아야 맞음)\n'
            'REUSE_EXPRS = [\n'
            '    U.Expr("a*a + a*b", lambda a, b: a * a + a * b, (1.5, -2.0)),\n'
            '    U.Expr("(a*b + b).tanh() * a", lambda a, b: (a * b + b).tanh() * a, (0.7, -1.3)),\n'
            '    U.Expr("((a + b) * (a * b)).tanh() + b*b", lambda a, b: ((a + b) * (a * b)).tanh() + b * b, (0.4, 0.9)),\n'
            ']\n'
            '# S13d: 빼기·나누기·거듭제곱·exp·숫자 섞기를 모두 쓴 식\n'
            'OPS_EXPRS = [\n'
            '    U.Expr("(a - b) / c**2", lambda a, b, c: (a - b) / c ** 2, (3.0, -1.5, 2.0)),\n'
            '    U.Expr("(2*a).exp() - b/3 + c**3", lambda a, b, c: (2 * a).exp() - b / 3 + c ** 3, (0.3, 1.2, -0.8)),\n'
            '    U.Expr("-a + b**0.5 * c - (a*c).exp()", lambda a, b, c: -a + b ** 0.5 * c - (a * c).exp(), (0.6, 2.0, -1.1)),\n'
            '    U.Expr("(e - 1)/(e + 1), e = (2*(a*b + c)).exp()", lambda a, b, c: _tanh_by_exp(a * b + c), (0.9, -0.4, 0.2)),\n'
            ']\n'
            '\n'
            '\n'
            'def _lecture_neuron(Value):\n'
            '    """강의 뉴런: x1 = 2, x2 = 0, w1 = -3, w2 = 1, b = 6.8813735870195432, n = x1*w1 + x2*w2 + b."""\n'
            '    r = ref()["S09"]\n'
            '    leaves = {k: U.make_value(Value, r[k]) for k in ("x1", "x2", "w1", "w2", "b")}\n'
            '    n = U.student("n = x1*w1 + x2*w2 + b 계산",\n'
            '                  lambda: leaves["x1"] * leaves["w1"] + leaves["x2"] * leaves["w2"] + leaves["b"])\n'
            '    U.require_value_like(n, "n = x1*w1 + x2*w2 + b 의 결과")\n'
            '    return leaves, n\n'
            '\n'
            '\n'
            'ORDER_HINT = ("결과 쪽부터 거꾸로 _backward()를 불러야 기울기가 재료로 전해집니다: "\n'
            '              "for node in reversed(topo): node._backward()")\n'
            'ROOT_HINT = "backward() 안에서 self.grad = 1.0 을 넣은 뒤 거꾸로 도세요."\n'
            '\n'
            '\n'
            'def _neuron_mismatch(leaves, n, o):\n'
            '    """강의 뉴런의 기울기를 손 계산 값과 비교. 다 맞으면 None, 다르면 (이름, 내 값, 맞는 값)."""\n'
            '    want = ref()["S11b"]["lecture_grads"]\n'
            '    t = tol()\n'
            '    for name, node in (("o", o), ("n", n)) + tuple((k, leaves[k]) for k in ("x1", "w1", "x2", "w2", "b")):\n'
            '        g = U.grad_of(node, name)\n'
            '        if not U.close(g, want[name], t):\n'
            '            return name, g, want[name]\n'
            '    return None\n'
            '\n'
            '\n'
            'def _neuron_message(mm, extra):\n'
            '    name, got, want = mm\n'
            '    if name == "o":\n'
            '        return f"backward() 뒤 o.grad: 내 값 {F(got)}, 맞는 값 1.0. {ROOT_HINT}"\n'
            '    tail = " (1 - o²)" if name == "n" else ""\n'
            '    return f"강의 뉴런의 {name}.grad: 내 값 {F(got)}, 맞는 값 {F(want)}{tail}. {extra}"\n'
            '\n'
            '\n'
            'def _ops_ok(Value, exprs):\n'
            '    """채점기가 직접 거꾸로 돌면(학생 backward() 없이) 맞는지. 맞으면 틀린 곳은 backward() 쪽이다."""\n'
            '    try:\n'
            '        leaves, n = _lecture_neuron(Value)\n'
            '        o = U.student("o = n.tanh() 계산", lambda: n.tanh())\n'
            '        U.require_value_like(o, "n.tanh() 의 결과")\n'
            '        U.drive_backward(o)\n'
            '        if _neuron_mismatch(leaves, n, o) is not None:\n'
            '            return False\n'
            '        U.compare_with_torch(Value, exprs, False, tol())\n'
            '        return True\n'
            '    except U.Fail:\n'
            '        return False\n'
            '\n'
            '\n'
            'def _why_backward_wrong(Value, exprs, ops_ref):\n'
            '    if _ops_ok(Value, exprs):\n'
            '        return ORDER_HINT\n'
            '    return f"연산마다의 _backward 가 틀렸습니다. {ops_ref} 를 먼저 맞추세요."\n'
            '\n'
            '\n'
            'def _check_add_mul_backward(Value, label):\n'
            '    """+ 와 * 의 _backward를 바로 불러 본다 (out.grad 배수 포함)."""\n'
            '    r = ref()["S10"]\n'
            '    t = tol()\n'
            '    hint_bw = "__init__에 self._backward = lambda: None 을 넣고, + 와 * 안에서 out._backward = _backward 로 넣으세요."\n'
            '    for og in r["out_grads"]:\n'
            '        a, b = U.make_value(Value, r["a"]), U.make_value(Value, r["b"])\n'
            '        c = U.student("c = a*b 계산", lambda: a * b)\n'
            '        U.require_value_like(c, "a*b 의 결과")\n'
            '        bw = U.get_attr(c, "_backward", "a*b 의 결과", hint=hint_bw)\n'
            '        U.set_grad(c, og, "c")\n'
            '        U.student("c._backward() 실행", bw)\n'
            '        ga, gb = U.grad_of(a, "a"), U.grad_of(b, "b")\n'
            '        wa, wb = r["b"] * og, r["a"] * og\n'
            '        if not (U.close(ga, wa, t) and U.close(gb, wb, t)):\n'
            '            extra = " 기울기가 0이면 out._backward = _backward 로 넣었는지 보세요." if ga == 0 and gb == 0 else ""\n'
            '            raise U.Fail(f"{label}곱셈 c = a*b (a = {F(r[\'a\'])}, b = {F(r[\'b\'])}, c.grad = {F(og)}) 뒤 "\n'
            '                         f"a.grad: 내 값 {F(ga)}, 맞는 값 {F(wa)} / b.grad: 내 값 {F(gb)}, 맞는 값 {F(wb)}. "\n'
            '                         f"곱셈은 상대 값을 곱해 전합니다: self.grad += other.data * out.grad, "\n'
            '                         f"other.grad += self.data * out.grad.{extra}")\n'
            '        a, b = U.make_value(Value, r["a"]), U.make_value(Value, r["b"])\n'
            '        d = U.student("d = a + b 계산", lambda: a + b)\n'
            '        U.require_value_like(d, "a + b 의 결과")\n'
            '        bw = U.get_attr(d, "_backward", "a + b 의 결과", hint=hint_bw)\n'
            '        U.set_grad(d, og, "d")\n'
            '        U.student("d._backward() 실행", bw)\n'
            '        ga, gb = U.grad_of(a, "a"), U.grad_of(b, "b")\n'
            '        if not (U.close(ga, og, t) and U.close(gb, og, t)):\n'
            '            raise U.Fail(f"{label}덧셈 d = a + b (d.grad = {F(og)}) 뒤 a.grad: 내 값 {F(ga)}, b.grad: 내 값 {F(gb)}, "\n'
            '                         f"맞는 값 둘 다 {F(og)}. 덧셈은 기울기를 그대로 전합니다: self.grad += 1.0 * out.grad, "\n'
            '                         f"other.grad += 1.0 * out.grad.")\n'
            '    leaf = U.make_value(Value, 1.0)\n'
            '    bw = U.get_attr(leaf, "_backward", "Value(1.0)", hint=hint_bw)\n'
            '    U.student("재료 값(Value(1.0))의 _backward() 실행", bw)\n'
            '\n'
            '\n'
            'def _check_tanh_backward(Value, label):\n'
            '    r = ref()["S10b"]\n'
            '    t = tol()\n'
            '    for x in r["x_values"]:\n'
            '        for og in r["out_grads"]:\n'
            '            v = U.make_value(Value, x)\n'
            '            o = U.student(f"Value({F(x)}).tanh() 계산", lambda: v.tanh())\n'
            '            U.require_value_like(o, "tanh() 의 결과")\n'
            '            bw = U.get_attr(o, "_backward", "tanh() 의 결과",\n'
            '                            hint="tanh 안에서 def _backward(): ... 를 만들고 out._backward = _backward 로 넣으세요.")\n'
            '            U.set_grad(o, og, "o")\n'
            '            U.student("o._backward() 실행", bw)\n'
            '            got = U.grad_of(v, "x")\n'
            '            th = math.tanh(x)\n'
            '            want = (1 - th * th) * og\n'
            '            if not U.close(got, want, t):\n'
            '                raise U.Fail(f"{label}o = Value({F(x)}).tanh(), o.grad = {F(og)} 뒤 x.grad: 내 값 {F(got)}, "\n'
            '                             f"맞는 값 {F(want)}. tanh의 기울기는 (1 - t**2) * out.grad 입니다 (t는 tanh 결과).")\n'
            '\n'
            '\n'
            '# ---- 단계 S01 ~ S16 ---------------------------------------------------------\n'
            '\n'
            '@grader("L01-S01", "함수 그래프 그리기", \'check("L01-S01", f)\')\n'
            'def s01(f=None, seen=False):\n'
            '    """f: 그래프를 그린 함수 f(x) = 3x² - 4x + 5. 그림만 눈으로 봤다면 check("L01-S01", seen=True)."""\n'
            '    if f is None and seen is True:\n'
            '        return "눈으로 확인했다고 표시했습니다. 그림이 포물선(U자 모양)이면 맞습니다."\n'
            '    U.need_callable(f, "f", \'check("L01-S01", f)\')\n'
            '    for x, want in ref()["S01"]["f_at"]:\n'
            '        got = U.as_number(U.student(f"f({F(x)}) 계산", f, x), f"f({F(x)})")\n'
            '        if not U.close(got, want, tol()):\n'
            '            raise U.Fail(f"f({F(x)}): 내 값 {F(got)}, 맞는 값 {F(want)}. f(x) = 3*x**2 - 4*x + 5 인지 보세요.")\n'
            '    return "f(x) = 3x² - 4x + 5 가 맞습니다. 그림은 포물선(U자 모양)이어야 합니다."\n'
            '\n'
            '\n'
            '@grader("L01-S02", "숫자 기울기", \'check("L01-S02", slope)\')\n'
            'def s02(slope=None):\n'
            '    """slope: (f(x + h) - f(x)) / h 로 구한 숫자 (x = 3.0, h = 0.0001이면 약 14)."""\n'
            '    v = U.as_number(slope, "기울기")\n'
            '    r = ref()["S02"]\n'
            '    if r["min"] <= v <= r["max"]:\n'
            '        return f"숫자 기울기 {F(v)}: 약 14가 맞습니다. x를 조금 올리면 f가 약 14배만큼 오릅니다."\n'
            '    if r["min"] <= -v <= r["max"]:\n'
            '        hint = "부호가 반대입니다. (f(x + h) - f(x)) / h 순서인지 보세요."\n'
            '    elif 0 < abs(v) < 0.01:\n'
            '        hint = "h로 나누는 것을 빠뜨렸나요?"\n'
            '    else:\n'
            '        hint = "x = 3.0, h = 0.0001 인지 보세요."\n'
            '    raise U.Fail(f"기울기: 내 값 {F(v)}, 맞는 값 약 14 ({F(r[\'min\'])}~{F(r[\'max\'])}). {hint}")\n'
            '\n'
            '\n'
            '@grader("L01-S03", "입력이 여러 개일 때", \'check("L01-S03", slope_a, slope_b=slope_b, slope_c=slope_c)\')\n'
            'def s03(slope_a=None, slope_b=None, slope_c=None):\n'
            '    """slope_a: a만 h만큼 올려 구한 (d2 - d1) / h (약 -3.0). 선택: slope_b(약 2.0), slope_c(약 1.0)."""\n'
            '    r = ref()["S03"]\n'
            '    want = {"a": -3.0, "b": 2.0, "c": 1.0}\n'
            '    hint = {"a": "a만 h만큼 올린 d2와 비교해 (d2 - d1) / h 를 구하세요. 답은 b의 값(-3.0)입니다.",\n'
            '            "b": "b만 h만큼 올리면 기울기는 a의 값(2.0)입니다.",\n'
            '            "c": "c만 h만큼 올리면 기울기는 1.0입니다."}\n'
            '    done = []\n'
            '    for name, val in (("a", slope_a), ("b", slope_b), ("c", slope_c)):\n'
            '        if val is None and name != "a":\n'
            '            continue\n'
            '        v = U.as_number(val, f"{name}에 대한 기울기")\n'
            '        if not r[name]["min"] <= v <= r[name]["max"]:\n'
            '            raise U.Fail(f"{name}에 대한 기울기: 내 값 {F(v)}, 맞는 값 약 {F(want[name])}. {hint[name]}")\n'
            '        done.append(f"{name} → {F(want[name])}")\n'
            '    return "d = a*b + c 의 숫자 기울기가 맞습니다: " + ", ".join(done) + ". d를 a로 미분하면 b의 값입니다."\n'
            '\n'
            '\n'
            '@grader("L01-S04", "Value 클래스 뼈대", \'check("L01-S04", Value)\')\n'
            'def s04(Value=None):\n'
            '    """Value: data, grad, _prev, _op, __repr__ 가 있는 클래스."""\n'
            '    usage = USAGE["L01-S04"]\n'
            '    U.need_callable(Value, "Value", usage)\n'
            '    r = ref()["S04"]\n'
            '    v = U.make_value(Value, r["data"])\n'
            '    d = U.data_of(v, "Value(2.0)")\n'
            '    if d != r["data"]:\n'
            '        raise U.Fail(f"Value(2.0).data: 내 값 {F(d)}, 맞는 값 2.0. self.data = data 인지 보세요.")\n'
            '    g = U.grad_of(v, "Value(2.0)")\n'
            '    if g != 0.0:\n'
            '        raise U.Fail(f"처음 grad: 내 값 {F(g)}, 맞는 값 0.0. self.grad = 0.0 으로 시작하세요.")\n'
            '    if U.prev_of(v, "Value(2.0)"):\n'
            '        raise U.Fail("재료 없이 만든 Value(2.0)의 _prev는 비어 있어야 합니다. self._prev = set(_children) 인지 보세요.")\n'
            '    op = U.get_attr(v, "_op", "Value(2.0)", hint="__init__에 self._op = _op 를 넣으세요.")\n'
            '    if not (isinstance(op, str) and op == ""):\n'
            '        raise U.Fail(f"재료 없이 만든 Value(2.0)의 _op: 내 값 {U.short(op)}, 맞는 값 \'\'(빈 글자).")\n'
            '    text = U.student("Value(2.0)의 출력 모양 만들기", repr, v)\n'
            '    if not (isinstance(text, str) and text.startswith(r["repr_prefix"])):\n'
            '        raise U.Fail(f"출력 모양: 내 값 {U.short(text)}, 맞는 값 \'Value(data=2.0)\'. "\n'
            '                     "__repr__에서 f\\"Value(data={self.data})\\" 를 돌려주세요.")\n'
            '    a, b = U.make_value(Value, 1.0), U.make_value(Value, 3.0)\n'
            '    c = U.student("Value(4.0, (a, b), \'+\') 만들기", Value, 4.0, (a, b), "+")\n'
            '    U.require_value_like(c, "Value(4.0, (a, b), \'+\')")\n'
            '    if not U.same_items(U.prev_of(c, "Value(4.0, (a, b), \'+\')"), [a, b]):\n'
            '        raise U.Fail("Value(4.0, (a, b), \'+\')의 _prev에 a와 b가 들어 있어야 합니다. self._prev = set(_children) 인지 보세요.")\n'
            '    op = U.get_attr(c, "_op", "Value(4.0, (a, b), \'+\')")\n'
            '    if not (isinstance(op, str) and op == "+"):\n'
            '        raise U.Fail(f"Value(4.0, (a, b), \'+\')의 _op: 내 값 {U.short(op)}, 맞는 값 \'+\'. self._op = _op 인지 보세요.")\n'
            '    return "Value(2.0)이 Value(data=2.0)으로 보이고, data·grad·_prev·_op 칸이 모두 맞습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S05", "덧셈과 곱셈 기록하기", \'check("L01-S05", Value)\')\n'
            'def s05(Value=None):\n'
            '    """Value: __add__, __mul__ 이 결과 Value에 재료 (self, other)와 기호 \'+\', \'*\' 를 넘기는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S05"])\n'
            '    r = ref()["S05"]\n'
            '    t = tol()\n'
            '    a, b, c = U.make_value(Value, r["a"]), U.make_value(Value, r["b"]), U.make_value(Value, r["c"])\n'
            '    d = U.student("d = a*b + c 계산", lambda: a * b + c)\n'
            '    U.require_value_like(d, "a*b + c 의 결과")\n'
            '    dd = U.data_of(d, "d")\n'
            '    if not U.close(dd, r["d"], t):\n'
            '        raise U.Fail(f"d = a*b + c (a = 2.0, b = -3.0, c = 10.0): 내 값 {F(dd)}, 맞는 값 4.0. "\n'
            '                     "__add__는 data를 더하고 __mul__은 곱하나요?")\n'
            '    op = U.get_attr(d, "_op", "d")\n'
            '    if not (isinstance(op, str) and op == "+"):\n'
            '        raise U.Fail(f"d._op: 내 값 {U.short(op)}, 맞는 값 \'+\'. __add__에서 Value(..., (self, other), \'+\') 처럼 넘기세요.")\n'
            '    prev = U.prev_of(d, "d")\n'
            '    if len(prev) != 2 or not any(p is c for p in prev):\n'
            '        raise U.Fail("d._prev에 재료 두 개(a*b의 결과와 c)가 있어야 합니다. Value(..., (self, other), \'+\') 처럼 넘기세요.")\n'
            '    e = [p for p in prev if p is not c][0]\n'
            '    ed = U.data_of(e, "a*b")\n'
            '    if not U.close(ed, r["ab"], t):\n'
            '        raise U.Fail(f"a*b: 내 값 {F(ed)}, 맞는 값 -6.0.")\n'
            '    op = U.get_attr(e, "_op", "a*b 의 결과")\n'
            '    if not (isinstance(op, str) and op == "*"):\n'
            '        raise U.Fail(f"(a*b)._op: 내 값 {U.short(op)}, 맞는 값 \'*\'. __mul__에서 Value(..., (self, other), \'*\') 처럼 넘기세요.")\n'
            '    if not U.same_items(U.prev_of(e, "a*b 의 결과"), [a, b]):\n'
            '        raise U.Fail("a*b 의 _prev에 a와 b가 있어야 합니다. Value(..., (self, other), \'*\') 처럼 넘기세요.")\n'
            '    return "d = a*b + c 가 4.0이고, d._prev에 재료 두 개, d._op에 \'+\'가 들어 있습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S06", "계산 그래프 그림", \'check("L01-S06", draw_dot(d))\')\n'
            'def s06(dot=None, seen=False):\n'
            '    """dot: draw_dot(d)가 돌려준 graphviz 그림. graphviz를 못 쓰는 곳이면 check("L01-S06", seen=True)."""\n'
            '    if dot is None and seen is True:\n'
            '        return "눈으로 확인했다고 표시했습니다. 상자(a, b, c, d)와 동그라미(+, *)가 이어져 있으면 맞습니다."\n'
            '    U.need(dot, "그림")\n'
            '    ok_msg = "graphviz 그림입니다. 상자(값)와 동그라미(+, *)가 이어졌는지 눈으로 확인하세요."\n'
            '    try:\n'
            '        module = type(dot).__module__\n'
            '    except Exception:\n'
            '        module = ""\n'
            '    if isinstance(module, str) and module.split(".")[0] == "graphviz":\n'
            '        return ok_msg\n'
            '    source = U.try_attr(dot, "source")\n'
            '    if isinstance(source, str) and "digraph" in source:\n'
            '        return ok_msg\n'
            '    raise U.Fail(f"graphviz 그림이 아닙니다(받은 것: {U.type_name(dot)}). check(\\"L01-S06\\", draw_dot(d)) 처럼 "\n'
            '                 "draw_dot의 결과를 넘기세요.")\n'
            '\n'
            '\n'
            '@grader("L01-S07", "손으로 기울기 채우기", \'check("L01-S07", a=a, b=b, c=c, d=d, e=e, f=f, L=L)\')\n'
            'def s07(a=None, b=None, c=None, d=None, e=None, f=None, L=None):\n'
            '    """e = a*b, d = e + c, L = d * f 로 만든 Value 7개. 각 grad를 손으로 채운 뒤 넘긴다."""\n'
            '    nodes = {"a": a, "b": b, "c": c, "d": d, "e": e, "f": f, "L": L}\n'
            '    for name, node in nodes.items():\n'
            '        U.require_value_like(node, name)\n'
            '    v = {name: U.data_of(node, name) for name, node in nodes.items()}\n'
            '    g = {name: U.grad_of(node, name) for name, node in nodes.items()}\n'
            '    t = tol()\n'
            '    if not (U.close(v["e"], v["a"] * v["b"], t) and U.close(v["d"], v["e"] + v["c"], t)\n'
            '            and U.close(v["L"], v["d"] * v["f"], t)):\n'
            '        raise U.Fail("식 모양이 다릅니다. e = a*b, d = e + c, L = d * f 로 만든 값을 넘기세요.")\n'
            '    want = {"L": 1.0, "f": v["d"], "d": v["f"]}\n'
            '    want["e"] = want["c"] = want["d"]\n'
            '    want["a"] = want["e"] * v["b"]\n'
            '    want["b"] = want["e"] * v["a"]\n'
            '    hints = {\n'
            '        "L": "맨 끝 L의 grad는 1.0부터 시작합니다.",\n'
            '        "f": "L = d * f 이므로 f.grad 는 d의 값입니다.",\n'
            '        "d": "L = d * f 이므로 d.grad 는 f의 값입니다.",\n'
            '        "e": "d = e + c 에서 + 노드는 기울기를 그대로 전합니다: e.grad = d.grad",\n'
            '        "c": "d = e + c 에서 + 노드는 기울기를 그대로 전합니다: c.grad = d.grad",\n'
            '        "a": "e = a * b 에서 * 노드는 상대 값을 곱해 전합니다: a.grad = e.grad * b.data",\n'
            '        "b": "e = a * b 에서 * 노드는 상대 값을 곱해 전합니다: b.grad = e.grad * a.data",\n'
            '    }\n'
            '    for name in ("L", "f", "d", "e", "c", "a", "b"):\n'
            '        if not U.close(g[name], want[name], t):\n'
            '            raise U.Fail(f"{name}.grad: 내 값 {F(g[name])}, 맞는 값 {F(want[name])}. {hints[name]}")\n'
            '    return "손으로 채운 기울기 7개가 모두 맞습니다. + 노드는 그대로, * 노드는 상대 값을 곱해 전했습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S08", "한 걸음 움직여 보기", \'check("L01-S08", L)\')\n'
            'def s08(L=None, L_after=None):\n'
            '    """L: S08 셀을 실행한 뒤의 L (Value 또는 L.data). 움직이기 전 값은 강의 숫자의 -8.0으로 본다.\n'
            '    숫자를 바꿔 실습했다면 check("L01-S08", L_before, L_after) 처럼 전·후 두 값을 넘긴다."""\n'
            '    r = ref()["S08"]\n'
            '    if L_after is None:\n'
            '        before, after = r["before"], U.as_number(L, "움직인 뒤 L")\n'
            '    else:\n'
            '        before, after = U.as_number(L, "움직이기 전 L"), U.as_number(L_after, "움직인 뒤 L")\n'
            '    if after == before:\n'
            '        raise U.Fail("L이 그대로입니다. a, b, c, f 를 움직인 뒤 e, d, L 을 다시 계산했나요?")\n'
            '    if not after > before:\n'
            '        raise U.Fail(f"L: {F(before)} → {F(after)}. 커져야 합니다. v.data += 0.01 * v.grad 처럼 기울기 방향(+)으로 "\n'
            '                     "움직였는지, S07에서 grad를 먼저 채웠는지 보세요.")\n'
            '    if before == r["before"] and not r["after_min"] <= after <= r["after_max"]:\n'
            '        raise U.Fail(f"움직인 뒤 L: 내 값 {F(after)}, 맞는 값 약 -7.29 ({F(r[\'after_min\'])}~{F(r[\'after_max\'])}). "\n'
            '                     "a, b, c, f 네 값을 0.01 * grad 만큼씩 움직였는지 보세요.")\n'
            '    return f"L이 커졌습니다: {F(before)} → {F(after)}. 기울기 방향으로 움직이면 결과가 커집니다."\n'
            '\n'
            '\n'
            '@grader("L01-S09", "뉴런 하나 계산하기", \'check("L01-S09", n, x1=x1, x2=x2, w1=w1, w2=w2, b=b)\')\n'
            'def s09(n=None, x1=None, x2=None, w1=None, w2=None, b=None):\n'
            '    """n = x1*w1 + x2*w2 + b 로 계산한 Value와, 그 재료 Value 5개."""\n'
            '    U.require_value_like(n, "n")\n'
            '    parts = {"x1": x1, "x2": x2, "w1": w1, "w2": w2, "b": b}\n'
            '    vals = {}\n'
            '    for name, node in parts.items():\n'
            '        U.require_value_like(node, name)\n'
            '        vals[name] = U.data_of(node, name)\n'
            '    want = vals["x1"] * vals["w1"] + vals["x2"] * vals["w2"] + vals["b"]\n'
            '    got = U.data_of(n, "n")\n'
            '    if not U.close(got, want, tol()):\n'
            '        raise U.Fail(f"n: 내 값 {F(got)}, 맞는 값 {F(want)}. n = x1*w1 + x2*w2 + b 인지 보세요.")\n'
            '    reach = {id(x) for x in U.walk(n, "n")}\n'
            '    missing = [name for name, node in parts.items() if id(node) not in reach]\n'
            '    if missing:\n'
            '        raise U.Fail(f"n의 재료에 없는 값: {\', \'.join(missing)}. 넘긴 Value들로 n = x1*w1 + x2*w2 + b 를 계산하세요.")\n'
            '    return f"n = x1*w1 + x2*w2 + b 계산이 맞습니다 (n = {F(got)}). 곱셈 두 개와 덧셈이 이어져 있습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S09b", "tanh 붙이기", \'check("L01-S09b", Value, o=o)\')\n'
            'def s09b(Value=None, o=None):\n'
            '    """Value: tanh() 메서드가 있는 클래스. 선택: o = n.tanh() 결과."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S09b"])\n'
            '    t = tol()\n'
            '    for x in ref()["S09b"]["x_values"]:\n'
            '        v = U.make_value(Value, x)\n'
            '        out = U.student(f"Value({F(x)}).tanh() 계산", lambda: v.tanh())\n'
            '        U.require_value_like(out, f"Value({F(x)}).tanh() 의 결과")\n'
            '        got = U.data_of(out, "tanh 결과")\n'
            '        want = math.tanh(x)\n'
            '        if not U.close(got, want, t):\n'
            '            raise U.Fail(f"Value({F(x)}).tanh(): 내 값 {F(got)}, 맞는 값 {F(want)}. "\n'
            '                         "math.tanh(x) 또는 (math.exp(2*x) - 1)/(math.exp(2*x) + 1) 를 쓰세요.")\n'
            '        if not U.same_items(U.prev_of(out, "tanh 결과"), [v]):\n'
            '            raise U.Fail("tanh 결과의 재료(_prev)는 자기 자신(self) 하나여야 합니다. Value(t, (self,), \'tanh\') 처럼 넘기세요.")\n'
            '    if o is not None:\n'
            '        U.require_value_like(o, "o")\n'
            '        od = U.data_of(o, "o")\n'
            '        if not -1 < od < 1:\n'
            '            raise U.Fail(f"o: 내 값 {F(od)}. tanh 결과는 -1과 1 사이여야 합니다.")\n'
            '    return "tanh 결과가 -1과 1 사이이고, 재료는 자기 자신 하나입니다. tanh의 기울기는 1 - o² 입니다."\n'
            '\n'
            '\n'
            '@grader("L01-S10", "_backward를 + 와 * 안에 넣기", \'check("L01-S10", Value)\')\n'
            'def s10(Value=None):\n'
            '    """Value: __init__에 self._backward, + 와 * 안에 _backward 가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S10"])\n'
            '    _check_add_mul_backward(Value, "")\n'
            '    return "+ 와 * 의 _backward가 기울기를 재료에게 맞게 나눠 줍니다(+ 는 그대로, * 는 상대 값을 곱해)."\n'
            '\n'
            '\n'
            '@grader("L01-S10b", "tanh의 _backward", \'check("L01-S10b", Value)\')\n'
            'def s10b(Value=None):\n'
            '    """Value: tanh 안에 _backward 가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S10b"])\n'
            '    _check_tanh_backward(Value, "")\n'
            '    # 교사 정본의 확인: 강의 뉴런에서 o.grad = 1.0; o._backward(); n._backward() 뒤 n.grad 0.5, b.grad 0.5\n'
            '    leaves, n = _lecture_neuron(Value)\n'
            '    o = U.student("o = n.tanh() 계산", lambda: n.tanh())\n'
            '    U.require_value_like(o, "n.tanh() 의 결과")\n'
            '    U.set_grad(o, 1.0, "o")\n'
            '    U.student("o._backward() 실행", U.get_attr(o, "_backward", "o"))\n'
            '    U.student("n._backward() 실행", U.get_attr(n, "_backward", "n"))\n'
            '    want = ref()["S11b"]["lecture_grads"]\n'
            '    for name, node in (("n", n), ("b", leaves["b"])):\n'
            '        g = U.grad_of(node, name)\n'
            '        if not U.close(g, want[name], tol()):\n'
            '            raise U.Fail(f"강의 뉴런에서 o._backward(), n._backward() 뒤 {name}.grad: 내 값 {F(g)}, 맞는 값 0.5. "\n'
            '                         "tanh 는 (1 - t**2) * out.grad, 덧셈은 out.grad 를 그대로 전합니다.")\n'
            '    return "tanh의 _backward가 (1 - t²) × out.grad 를 전합니다. 강의 뉴런의 n.grad, b.grad 도 0.5 입니다."\n'
            '\n'
            '\n'
            '@grader("L01-S11", "위상 정렬", \'check("L01-S11", topo, o)\')\n'
            'def s11(topo=None, o=None):\n'
            '    """topo: build_topo(o)로 만든 Value 목록, o: 맨 끝 값."""\n'
            '    U.need(topo, "topo 목록")\n'
            '    U.require_value_like(o, "o")\n'
            '    if isinstance(topo, (str, bytes)):\n'
            '        raise U.Fail("topo 에는 Value 목록이 와야 합니다.")\n'
            '    items = U.student("topo 목록 읽기", list, topo)\n'
            '    if not items:\n'
            '        raise U.Fail("topo 목록이 비어 있습니다. build_topo(o)를 실행한 뒤 topo를 넘기세요.")\n'
            '    for i, item in enumerate(items):\n'
            '        U.require_value_like(item, f"topo[{i}]")\n'
            '    reach_ids = {id(x) for x in U.walk(o, "o")}\n'
            '    ids = [id(x) for x in items]\n'
            '    if len(set(ids)) != len(ids):\n'
            '        raise U.Fail("같은 값이 목록에 두 번 이상 들어 있습니다. visited에 이미 넣은 값은 건너뛰세요.")\n'
            '    missing = reach_ids - set(ids)\n'
            '    if missing:\n'
            '        raise U.Fail(f"o에 이어진 값 {len(missing)}개가 목록에 없습니다. 재료(_prev)를 모두 따라가며 넣으세요.")\n'
            '    extra = set(ids) - reach_ids\n'
            '    if extra:\n'
            '        raise U.Fail(f"o와 이어지지 않은 값 {len(extra)}개가 목록에 있습니다. build_topo(o)로 새로 만든 목록인지 보세요.")\n'
            '    if ids[-1] != id(o):\n'
            '        raise U.Fail("맨 끝이 o가 아닙니다. 재료를 먼저 넣고, 결과(o)는 맨 뒤에 와야 합니다.")\n'
            '    pos = {i: k for k, i in enumerate(ids)}\n'
            '    for node in items:\n'
            '        for child in U.prev_of(node, "값"):\n'
            '            if pos[id(child)] > pos[id(node)]:\n'
            '                raise U.Fail("재료가 결과보다 뒤에 있습니다. 재료(_prev)를 다 넣은 다음에 자기 자신을 넣으세요 "\n'
            '                             "(topo.append(v)는 for 문 뒤).")\n'
            '    return f"값 {len(items)}개가 재료부터 결과 순으로 줄 섰습니다. 맨 끝은 o입니다."\n'
            '\n'
            '\n'
            '@grader("L01-S11b", "backward() 메서드", \'check("L01-S11b", Value)\')\n'
            'def s11b(Value=None):\n'
            '    """Value: backward() 메서드(위상 정렬 + self.grad = 1.0 + reversed)가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S11b"])\n'
            '    leaves, n = _lecture_neuron(Value)\n'
            '    o = U.student("o = n.tanh() 계산", lambda: n.tanh())\n'
            '    U.require_value_like(o, "n.tanh() 의 결과")\n'
            '    bw = U.get_attr(o, "backward", "Value", hint="Value에 backward() 메서드를 넣으세요.")\n'
            '    U.student("o.backward() 실행", bw)\n'
            '    ops_ref = "L01-S10(+, *)과 L01-S10b(tanh)"\n'
            '    mm = _neuron_mismatch(leaves, n, o)\n'
            '    if mm is not None:\n'
            '        extra = "" if mm[0] == "o" else _why_backward_wrong(Value, TREE_EXPRS, ops_ref)\n'
            '        raise U.Fail(_neuron_message(mm, extra))\n'
            '    try:\n'
            '        done = U.compare_with_torch(Value, TREE_EXPRS, True, tol())\n'
            '    except U.Fail as exc:\n'
            '        raise U.Fail(f"{exc.message_ko} {_why_backward_wrong(Value, TREE_EXPRS, ops_ref)}") from None\n'
            '    return "o.backward() 한 줄로 손으로 구한 기울기와 같은 값이 나옵니다." + U.torch_note(done, len(TREE_EXPRS))\n'
            '\n'
            '\n'
            '@grader("L01-S12", "같은 값을 두 번 쓰는 버그", \'check("L01-S12", Value)\')\n'
            'def s12(Value=None):\n'
            '    """Value: += 로 기울기를 쌓는 클래스. 또는 a = Value(3.0); b = a + a; b.backward() 뒤의 a.grad 숫자."""\n'
            '    U.need(Value, "Value")\n'
            '    r = ref()["S12"]\n'
            '    t = tol()\n'
            '    if not callable(Value):\n'
            '        g = U.as_number(Value, "a.grad")\n'
            '        if U.close(g, r["a_plus_a"], t):\n'
            '            return "a.grad = 2.0 이 맞습니다. 두 길의 기울기를 += 로 더했습니다."\n'
            '        if U.close(g, 1.0, t):\n'
            '            raise U.Fail("a.grad: 내 값 1.0, 맞는 값 2.0. _backward 안에서 = 대신 += 로 기울기를 쌓아야 합니다.")\n'
            '        raise U.Fail(f"a.grad: 내 값 {F(g)}, 맞는 값 2.0. a = Value(3.0); b = a + a; b.backward() 뒤의 a.grad를 넘기세요.")\n'
            '    a = U.make_value(Value, r["a"])\n'
            '    b = U.student("b = a + a 계산", lambda: a + a)\n'
            '    U.require_value_like(b, "a + a 의 결과")\n'
            '    U.drive_backward(b)\n'
            '    g = U.grad_of(a, "a")\n'
            '    if not U.close(g, r["a_plus_a"], t):\n'
            '        raise U.Fail(f"b = a + a 뒤 a.grad: 내 값 {F(g)}, 맞는 값 2.0. 같은 값이 두 길로 쓰이면 기울기를 더해야 합니다. "\n'
            '                     "_backward 안의 = 를 += 로 바꾸세요.")\n'
            '    a = U.make_value(Value, r["a"])\n'
            '    b = U.student("b = a * a 계산", lambda: a * a)\n'
            '    U.require_value_like(b, "a * a 의 결과")\n'
            '    U.drive_backward(b)\n'
            '    g = U.grad_of(a, "a")\n'
            '    if not U.close(g, r["a_times_a"], t):\n'
            '        raise U.Fail(f"b = a * a 뒤 a.grad: 내 값 {F(g)}, 맞는 값 6.0 (2 × a). 곱셈의 _backward 를 보세요(L01-S10): "\n'
            '                     "self.grad += other.data * out.grad, other.grad += self.data * out.grad 둘 다 += 로 쌓아야 합니다.")\n'
            '    return "같은 값을 두 번 써도 기울기가 += 로 쌓입니다 (a + a 는 2.0, a * a 는 6.0)."\n'
            '\n'
            '\n'
            '# S13 ~ S13c 의 연산 하나짜리 식. 숫자는 reference/l01.json 의 cases 에 있고, 식은 여기 이름으로 찾는다.\n'
            '_ONE_OP = {\n'
            '    "a + 1": lambda a, b: a + 1,\n'
            '    "1 + a": lambda a, b: 1 + a,\n'
            '    "a * 2": lambda a, b: a * 2,\n'
            '    "2 * a": lambda a, b: 2 * a,\n'
            '    "a.exp()": lambda a, b: a.exp(),\n'
            '    "-a": lambda a, b: -a,\n'
            '    "a - b": lambda a, b: a - b,\n'
            '    "a - 1": lambda a, b: a - 1,\n'
            '    "a ** 2": lambda a, b: a ** 2,\n'
            '    "a ** -1": lambda a, b: a ** -1,\n'
            '    "a ** 0.5": lambda a, b: a ** 0.5,\n'
            '    "a / 2": lambda a, b: a / 2,\n'
            '    "a / b": lambda a, b: a / b,\n'
            '}\n'
            '_POW_HINT = "거듭제곱의 기울기는 other * self.data**(other - 1) * out.grad 입니다(__pow__)."\n'
            '_ONE_OP_HINT = {\n'
            '    "a + 1": U._WRAP_HINT,\n'
            '    "1 + a": "숫자 + Value 는 __radd__ 가 필요합니다: def __radd__(self, other): return self + other",\n'
            '    "a * 2": U._WRAP_HINT,\n'
            '    "2 * a": "숫자 * Value 는 __rmul__ 이 필요합니다: def __rmul__(self, other): return self * other",\n'
            '    "a.exp()": "exp 는 Value(math.exp(self.data), (self,), \'exp\') 이고 기울기는 out.data * out.grad 입니다.",\n'
            '    "-a": "__neg__ 는 self * -1 입니다.",\n'
            '    "a - b": "빼기는 self + (-other) 로 만듭니다(__neg__, __sub__).",\n'
            '    "a - 1": "빼기는 self + (-other) 로 만듭니다. 숫자를 Value로 감싸는 줄도 필요합니다.",\n'
            '    "a ** 2": _POW_HINT,\n'
            '    "a ** -1": _POW_HINT,\n'
            '    "a ** 0.5": _POW_HINT,\n'
            '    "a / 2": "나누기는 self * other**-1 로 만듭니다(__truediv__).",\n'
            '    "a / b": "나누기는 self * other**-1 로 만듭니다(__truediv__). Value ** -1 이 필요해 __pow__ 도 씁니다.",\n'
            '}\n'
            '\n'
            '\n'
            'def _check_one_op_cases(Value, cases):\n'
            '    """cases: [{"expr", "a", "b"(선택), "value", "grad_a", "grad_b"(선택)}]. 값과 기울기를 손 계산 값과 비교."""\n'
            '    t = tol()\n'
            '    for case in cases:\n'
            '        expr = case["expr"]\n'
            '        fn = _ONE_OP[expr]\n'
            '        a = U.make_value(Value, case["a"])\n'
            '        b = U.make_value(Value, case["b"]) if "b" in case else None\n'
            '        where = f"a = {F(case[\'a\'])}" + (f", b = {F(case[\'b\'])}" if "b" in case else "")\n'
            '        out = U.student(f"{expr} ({where}) 계산", fn, a, b)\n'
            '        U.require_value_like(out, f"{expr} 의 결과")\n'
            '        got = U.data_of(out, expr)\n'
            '        if not U.close(got, case["value"], t):\n'
            '            raise U.Fail(f"{expr} ({where}): 내 값 {F(got)}, 맞는 값 {F(case[\'value\'])}. {_ONE_OP_HINT[expr]}")\n'
            '        U.drive_backward(out)\n'
            '        grads = [("a", a, case["grad_a"])] + ([("b", b, case["grad_b"])] if b is not None else [])\n'
            '        for name, node, want in grads:\n'
            '            g = U.grad_of(node, name)\n'
            '            if not U.close(g, want, t):\n'
            '                raise U.Fail(f"{expr} ({where}) 의 기울기 {name}.grad: 내 값 {F(g)}, 맞는 값 {F(want)}. {_ONE_OP_HINT[expr]}")\n'
            '\n'
            '\n'
            '@grader("L01-S13", "숫자와 섞어 계산하기", \'check("L01-S13", Value)\')\n'
            'def s13(Value=None):\n'
            '    """Value: 숫자를 Value로 감싸는 줄(__add__, __mul__ 첫 줄)과 __radd__, __rmul__ 이 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S13"])\n'
            '    r = ref()["S13"]\n'
            '    _check_one_op_cases(Value, r["cases"])\n'
            '    vals = [U.make_value(Value, v) for v in r["sum"]["values"]]\n'
            '    total = U.student("sum([Value(1.0), Value(2.0), Value(3.0)]) 계산", sum, vals)\n'
            '    U.require_value_like(total, "sum(...) 의 결과")\n'
            '    got = U.data_of(total, "sum(...)")\n'
            '    if not U.close(got, r["sum"]["value"], tol()):\n'
            '        raise U.Fail(f"sum([Value(1.0), Value(2.0), Value(3.0)]): 내 값 {F(got)}, 맞는 값 6.0. "\n'
            '                     "sum() 은 0부터 더하므로 __radd__ 가 필요합니다.")\n'
            '    return "Value와 숫자를 섞어 계산할 수 있습니다(Value(2.0) + 1, 1 + Value(2.0), 2 * Value(3.0)). sum() 도 됩니다."\n'
            '\n'
            '\n'
            '@grader("L01-S13b", "exp와 빼기", \'check("L01-S13b", Value)\')\n'
            'def s13b(Value=None):\n'
            '    """Value: exp(와 그 _backward), __neg__, __sub__ 가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S13b"])\n'
            '    _check_one_op_cases(Value, ref()["S13b"]["cases"])\n'
            '    return "exp 와 빼기(-a, a - b, a - 1)의 값과 기울기가 맞습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S13c", "거듭제곱과 나누기", \'check("L01-S13c", Value)\')\n'
            'def s13c(Value=None):\n'
            '    """Value: __pow__(와 그 _backward), __truediv__ 가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S13c"])\n'
            '    _check_one_op_cases(Value, ref()["S13c"]["cases"])\n'
            '    return "거듭제곱(a ** 2, a ** -1, a ** 0.5)과 나누기(a / 2, a / b)의 값과 기울기가 맞습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S13d", "tanh를 쪼개서 다시 계산하기", \'check("L01-S13d", Value)\')\n'
            'def s13d(Value=None):\n'
            '    """Value: exp, 빼기, 나누기, 숫자 섞기가 있는 클래스. 강의 뉴런을 o = (e - 1)/(e + 1) 로 계산해 본다."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-S13d"])\n'
            '    t = tol()\n'
            '    leaves, n = _lecture_neuron(Value)\n'
            '    o = U.student("e = (2*n).exp(); o = (e - 1) / (e + 1) 계산", _tanh_by_exp, n)\n'
            '    U.require_value_like(o, "(e - 1) / (e + 1) 의 결과")\n'
            '    od = U.data_of(o, "o")\n'
            '    want_o = ref()["S09b"]["o_lecture"]\n'
            '    if not U.close(od, want_o, t):\n'
            '        raise U.Fail(f"o: 내 값 {F(od)}, 맞는 값 {F(want_o)}. e = (2*n).exp(), o = (e - 1) / (e + 1) 인지 보세요.")\n'
            '    U.drive_backward(o)\n'
            '    mm = _neuron_mismatch(leaves, n, o)\n'
            '    if mm is not None:\n'
            '        raise U.Fail(_neuron_message(mm, "exp, 나누기, 빼기, 더하기의 _backward 를 하나씩 보세요(L01-S13b, L01-S13c). "\n'
            '                                         "식에서 e 를 두 번 쓰므로 += 로 쌓는지도 보세요."))\n'
            '    done = U.compare_with_torch(Value, OPS_EXPRS, False, t)\n'
            '    return ("tanh 를 exp, 빼기, 나누기로 풀어 써도 o = 0.7071, 기울기도 S11b 와 같습니다."\n'
            '            + U.torch_note(done, len(OPS_EXPRS)))\n'
            '\n'
            '\n'
            '@grader("L01-S13e", "PyTorch로 비교", \'check("L01-S13e", x1=x1, x2=x2, w1=w1, w2=w2, b=b, o=o)\')\n'
            'def s13e(x1=None, x2=None, w1=None, w2=None, b=None, o=None):\n'
            '    """x1, x2, w1, w2, b: requires_grad=True 인 PyTorch 텐서(o = torch.tanh(x1*w1 + x2*w2 + b); o.backward() 뒤). o는 선택."""\n'
            '    torch = U.get_torch()\n'
            '    if torch is None:\n'
            '        raise U.Skip("torch가 없어 건너뜀. PyTorch가 있는 곳(Colab 등)에서 다시 실행하면 채점됩니다.")\n'
            '    names = ("x1", "x2", "w1", "w2", "b")\n'
            '    tensors = dict(zip(names, (x1, x2, w1, w2, b)))\n'
            '    vals, grads, low = {}, {}, False\n'
            '    for name in names:\n'
            '        tensor = U.need(tensors[name], name)\n'
            '        if not isinstance(tensor, torch.Tensor):\n'
            '            raise U.Fail(f"{name} 자리에 PyTorch 텐서가 와야 합니다(받은 것: {U.type_name(tensor)}). "\n'
            '                         f"torch.tensor([2.0], dtype=torch.double, requires_grad=True) 처럼 만드세요.")\n'
            '        grad = U.get_attr(tensor, "grad", name)\n'
            '        if grad is None:\n'
            '            raise U.Fail(f"{name}.grad 가 비어 있습니다. requires_grad=True 로 만들고 o.backward() 를 실행했나요?")\n'
            '        if tensor.dtype != torch.float64:\n'
            '            low = True\n'
            '        vals[name] = U.as_number(tensor, name)\n'
            '        grads[name] = U.as_number(grad, f"{name}.grad")\n'
            '    n = vals["x1"] * vals["w1"] + vals["x2"] * vals["w2"] + vals["b"]\n'
            '    th = math.tanh(n)\n'
            '    g = 1 - th * th\n'
            '    want = {"x1": vals["w1"] * g, "w1": vals["x1"] * g, "x2": vals["w2"] * g, "w2": vals["x2"] * g, "b": g}\n'
            '    t = ref()["tolerance"]["float32"] if low else tol()\n'
            '    for name in ("x2", "w2", "x1", "w1", "b"):\n'
            '        if not U.close(grads[name], want[name], t):\n'
            '            raise U.Fail(f"{name}.grad: 내 값 {F(grads[name])}, 맞는 값 {F(want[name])}. "\n'
            '                         "o = torch.tanh(x1*w1 + x2*w2 + b), o.backward() 순서인지 보세요.")\n'
            '    if o is not None:\n'
            '        od = U.as_number(o, "o")\n'
            '        if not U.close(od, th, t):\n'
            '            raise U.Fail(f"o: 내 값 {F(od)}, 맞는 값 {F(th)}. o = torch.tanh(x1*w1 + x2*w2 + b) 인지 보세요.")\n'
            '    note = (" (float32 텐서라 소수점 넷째 자리까지만 비교했습니다. dtype=torch.double 로 만들면 micrograd와 소수점 아래까지 같아집니다.)"\n'
            '            if low else "")\n'
            '    return "PyTorch로 구한 기울기가 micrograd와 같습니다. 우리가 만든 것이 PyTorch가 하는 일의 핵심입니다." + note\n'
            '\n'
            '\n'
            '@grader("L01-S14", "Neuron 클래스", \'check("L01-S14", Neuron)\')\n'
            'def s14(Neuron=None):\n'
            '    """Neuron: Neuron(입력 개수)로 만들고, neuron(x 목록)이 -1~1 사이 Value를 내는 클래스."""\n'
            '    U.need_callable(Neuron, "Neuron", USAGE["L01-S14"])\n'
            '    r = ref()["S14"]\n'
            '    t = tol()\n'
            '    for case in r["cases"]:\n'
            '        nin, x = case["nin"], case["x"]\n'
            '        label = f"Neuron({nin})({x})"\n'
            '        for _ in range(r["repeat"]):\n'
            '            neuron = U.student(f"Neuron({nin}) 만들기", Neuron, nin)\n'
            '            out = U.student(f"{label} 계산", neuron, list(x))\n'
            '            U.require_value_like(out, f"{label} 의 결과")\n'
            '            od = U.data_of(out, label)\n'
            '            if not -1 < od < 1:\n'
            '                raise U.Fail(f"{label}: 내 값 {F(od)}. -1과 1 사이여야 합니다. 마지막에 tanh() 를 했나요?")\n'
            '            w, bias = U.try_attr(neuron, "w"), U.try_attr(neuron, "b")\n'
            '            if w is U.MISSING or bias is U.MISSING:\n'
            '                continue\n'
            '            ws = U.student("neuron.w 읽기", list, w)\n'
            '            if len(ws) != nin:\n'
            '                raise U.Fail(f"Neuron({nin}) 의 w 개수: 내 값 {len(ws)}개, 맞는 값 {nin}개. 입력마다 가중치가 하나씩 있어야 합니다.")\n'
            '            act = sum(U.data_of(wi, "w") * xi for wi, xi in zip(ws, x)) + U.data_of(bias, "b")\n'
            '            want = math.tanh(act)\n'
            '            if not U.close(od, want, t):\n'
            '                raise U.Fail(f"{label}: 내 값 {F(od)}, 맞는 값 {F(want)} (tanh(w·x + b)). "\n'
            '                             "w와 x를 곱해 더하고 b를 더한 뒤 tanh() 를 했나요?")\n'
            '    return "Neuron(2)([2.0, 3.0]) 이 -1과 1 사이 Value를 냅니다. tanh(w·x + b) 계산도 맞습니다."\n'
            '\n'
            '\n'
            '@grader("L01-S14b", "Layer 클래스", \'check("L01-S14b", Layer)\')\n'
            'def s14b(Layer=None):\n'
            '    """Layer: Layer(입력, 출력)로 만들고, layer(x 목록)이 Value 목록(출력 개수만큼)을 내는 클래스."""\n'
            '    U.need_callable(Layer, "Layer", USAGE["L01-S14b"])\n'
            '    for case in ref()["S14b"]["cases"]:\n'
            '        nin, nout, x = case["nin"], case["nout"], case["x"]\n'
            '        label = f"Layer({nin}, {nout})({x})"\n'
            '        layer = U.student(f"Layer({nin}, {nout}) 만들기", Layer, nin, nout)\n'
            '        outs = U.student(f"{label} 계산", layer, list(x))\n'
            '        U.need(outs, f"{label} 의 결과")\n'
            '        if U.is_value_like(outs) or isinstance(outs, (str, bytes)):\n'
            '            raise U.Fail(f"{label} 의 결과가 목록이 아닙니다. 뉴런 {nout}개의 결과를 목록으로 돌려주세요.")\n'
            '        items = U.student(f"{label} 의 결과 읽기", list, outs)\n'
            '        if len(items) != nout:\n'
            '            raise U.Fail(f"{label} 의 결과 개수: 내 값 {len(items)}개, 맞는 값 {nout}개. 뉴런을 출력 개수만큼 만드세요.")\n'
            '        for i, item in enumerate(items):\n'
            '            U.require_value_like(item, f"{label}[{i}]")\n'
            '            d = U.data_of(item, f"{label}[{i}]")\n'
            '            if not -1 < d < 1:\n'
            '                raise U.Fail(f"{label}[{i}]: 내 값 {F(d)}. 뉴런 결과는 -1과 1 사이여야 합니다.")\n'
            '    return "Layer(2, 3)([2.0, 3.0]) 이 Value 3개를 냅니다. 층은 같은 입력을 보는 뉴런 묶음입니다."\n'
            '\n'
            '\n'
            'def _param_breakdown(nin, nouts):\n'
            '    sizes = [nin] + list(nouts)\n'
            '    return ", ".join(f"{a}→{b}: {(a + 1) * b}개" for a, b in zip(sizes, sizes[1:]))\n'
            '\n'
            '\n'
            'def _parameters(model, label):\n'
            '    fn = U.get_attr(model, "parameters", label, hint="Neuron, Layer, MLP 세 클래스 모두에 parameters() 를 넣으세요.")\n'
            '    params = U.student(f"{label}.parameters() 실행", fn)\n'
            '    U.need(params, f"{label}.parameters() 의 결과")\n'
            '    return U.student(f"{label}.parameters() 의 결과 읽기", list, params)\n'
            '\n'
            '\n'
            '@grader("L01-S14c", "MLP 클래스와 parameters()", \'check("L01-S14c", MLP)\')\n'
            'def s14c(MLP=None):\n'
            '    """MLP: MLP(입력, [층별 출력])로 만들고 parameters() 가 모든 w, b 를 돌려주는 클래스."""\n'
            '    U.need_callable(MLP, "MLP", USAGE["L01-S14c"])\n'
            '    r = ref()["S14c"]\n'
            '    for case in r["cases"]:\n'
            '        nin, nouts, want = case["nin"], case["nouts"], case["count"]\n'
            '        label = f"MLP({nin}, {nouts})"\n'
            '        model = U.student(f"{label} 만들기", MLP, nin, list(nouts))\n'
            '        params = _parameters(model, label)\n'
            '        if len(params) != want:\n'
            '            if len(params) == want - sum(nouts):\n'
            '                extra = " 뉴런 수만큼 모자랍니다. Neuron.parameters() 가 w와 b를 모두 돌려주나요? (return self.w + [self.b])"\n'
            '            else:\n'
            '                extra = " Neuron, Layer, MLP 세 클래스 모두에 parameters() 가 있고, 층마다 다 모으나요?"\n'
            '            raise U.Fail(f"{label} 의 parameters() 개수: 내 값 {len(params)}개, 맞는 값 {want}개 "\n'
            '                         f"({_param_breakdown(nin, nouts)}).{extra}")\n'
            '        if len({id(p) for p in params}) != len(params):\n'
            '            raise U.Fail(f"{label}.parameters() 에 같은 값이 두 번 들어 있습니다.")\n'
            '        for i, p in enumerate(params):\n'
            '            U.require_value_like(p, f"{label}.parameters()[{i}]")\n'
            '    model = U.student("MLP(3, [4, 4, 1]) 만들기", MLP, 3, [4, 4, 1])\n'
            '    x = r["x"]\n'
            '    out = U.student(f"MLP(3, [4, 4, 1])({x}) 계산", model, list(x))\n'
            '    U.need(out, "MLP 출력")\n'
            '    if not U.is_value_like(out) and not isinstance(out, (str, bytes)):\n'
            '        items = U.student("MLP 출력 읽기", list, out)\n'
            '        if len(items) != 1:\n'
            '            raise U.Fail(f"MLP(3, [4, 4, 1]) 의 출력 개수: 내 값 {len(items)}개, 맞는 값 1개.")\n'
            '        out = items[0]\n'
            '    U.require_value_like(out, "MLP(3, [4, 4, 1]) 의 출력")\n'
            '    d = U.data_of(out, "MLP 출력")\n'
            '    if not -1 < d < 1:\n'
            '        raise U.Fail(f"MLP(3, [4, 4, 1]) 의 출력: 내 값 {F(d)}. -1과 1 사이여야 합니다.")\n'
            '    return "MLP(3, [4, 4, 1]) 의 파라미터가 41개입니다(3→4: 16개, 4→4: 20개, 4→1: 5개). 출력도 -1과 1 사이 Value입니다."\n'
            '\n'
            '\n'
            'def _model_loss(model, label):\n'
            '    """S15 데이터(xs, ys)로 손실 = (예측 - 정답)² 의 합을 채점기가 숫자로 다시 계산한다(학생 forward 만 씀)."""\n'
            '    data = ref()["S15"]\n'
            '    total = 0.0\n'
            '    for x, y in zip(data["xs"], data["ys"]):\n'
            '        out = U.student(f"{label}({x}) 계산", model, list(x))\n'
            '        U.need(out, f"{label} 출력")\n'
            '        if not U.is_value_like(out) and not isinstance(out, (str, bytes)):\n'
            '            items = U.student(f"{label} 출력 읽기", list, out)\n'
            '            if len(items) != 1:\n'
            '                raise U.Fail(f"{label} 출력 개수: 내 값 {len(items)}개, 맞는 값 1개.")\n'
            '            out = items[0]\n'
            '        total += (U.as_number(out, f"{label} 출력") - y) ** 2\n'
            '    return total\n'
            '\n'
            '\n'
            'def _check_mlp_41(model, label):\n'
            '    U.need(model, label)\n'
            '    want = ref()["S15"]["params"]\n'
            '    params = _parameters(model, label)\n'
            '    if len(params) != want:\n'
            '        raise U.Fail(f"{label} 의 parameters() 개수: 내 값 {len(params)}개, 맞는 값 {want}개 "\n'
            '                     f"({_param_breakdown(3, [4, 4, 1])}). {label} = MLP(3, [4, 4, 1]) 인지, "\n'
            '                     "Neuron.parameters() 가 w와 b를 모두 돌려주는지 보세요.")\n'
            '\n'
            '\n'
            'def _loss_number(loss, label):\n'
            '    v = U.as_number(loss, label)\n'
            '    if v < 0:\n'
            '        raise U.Fail(f"{label}: 내 값 {F(v)}. 손실은 0보다 작을 수 없습니다. (yout - ygt)**2 를 더했나요?")\n'
            '    if v >= ref()["S15"]["loss_upper_bound"]:\n'
            '        raise U.Fail(f"{label}: 내 값 {F(v)}. 데이터 4개, 정답 ±1, tanh 출력이면 손실은 16보다 작아야 합니다. "\n'
            '                     "S15의 xs, ys 와 MLP(3, [4, 4, 1]) 를 썼나요?")\n'
            '    return v\n'
            '\n'
            '\n'
            'def _check_trained(n, loss, limit, more):\n'
            '    """학습 뒤 n 과 마지막 loss. 마지막 손실과, 넘긴 n 으로 다시 계산한 손실이 둘 다 limit 미만이어야 한다."""\n'
            '    _check_mlp_41(n, "n")\n'
            '    last = _loss_number(loss, "마지막 loss")\n'
            '    if not last < limit:\n'
            '        raise U.Fail(f"마지막 손실: 내 값 {F(last)}, 기준 {F(limit)} 미만. {more} 손실이 줄지 않고 커지면 갱신이 "\n'
            '                     "p.data += -0.05 * p.grad (빼기)인지, p.grad = 0.0 이 loss.backward() 앞에 있는지 보세요.")\n'
            '    now = _model_loss(n, "n")\n'
            '    if not now < limit:\n'
            '        raise U.Fail(f"넘긴 n 으로 다시 계산한 손실: {F(now)}. {F(limit)} 미만이어야 합니다. 학습한 그 n 과 마지막 loss 를 함께 넘기세요.")\n'
            '    return last\n'
            '\n'
            '\n'
            '_S15_START = {}  # id(n) -> (n, S15 칸에서 통과한 시작 손실). S15b 가 "줄었는지" 비교에 쓴다\n'
            '\n'
            '\n'
            '@grader("L01-S15", "데이터와 손실", \'check("L01-S15", n, loss)\')\n'
            'def s15(n=None, loss=None):\n'
            '    """n: MLP(3, [4, 4, 1]) (학습 전), loss: S15 데이터로 계산한 손실(Value 또는 숫자). 학습(S15b) 전에 채점한다.\n'
            '    통과하면 이 n 의 시작 손실을 기억해 두었다가 L01-S15b 에서 손실이 줄었는지 비교한다."""\n'
            '    _check_mlp_41(n, "n")\n'
            '    lv = _loss_number(loss, "loss")\n'
            '    again = _model_loss(n, "n")\n'
            '    if not U.close(lv, again, tol()):\n'
            '        raise U.Fail(f"loss: 내 값 {F(lv)}, 넘긴 n 으로 다시 계산한 값 {F(again)}. "\n'
            '                     "loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred)) 인지, xs·ys 가 S15 데이터인지 보세요. "\n'
            '                     "학습(S15b) 전에 바로 채점해야 합니다.")\n'
            '    _S15_START[id(n)] = (n, lv)\n'
            '    return (f"파라미터 41개, 손실 {F(lv)}: 예측과 정답 차이를 제곱해 더한 값입니다"\n'
            '            "(시작값이 무작위라 실행마다 다르고 보통 1에서 8 사이).")\n'
            '\n'
            '\n'
            '@grader("L01-S15b", "학습 반복", \'check("L01-S15b", n, loss)\')\n'
            'def s15b(n=None, loss=None, start=None):\n'
            '    """n: 학습한 MLP(3, [4, 4, 1]), loss: 학습 셀이 마지막으로 계산한 손실(Value 또는 숫자).\n'
            '    시작 손실은 같은 n 으로 통과한 L01-S15 칸이 기억한 값을 쓴다. 그 칸을 거치지 않았다면 start=처음 손실 로 넘긴다.\n'
            '    교사 정본에 시드 고정이 없어 손실이 줄었는지만 본다(reference S15b.last_loss_max 가 null 이 아니면 그 기준도 본다)."""\n'
            '    _check_mlp_41(n, "n")\n'
            '    _loss_number(loss, "마지막 loss")\n'
            '    if start is not None:\n'
            '        first = _loss_number(start, "start(처음 손실)")\n'
            '    else:\n'
            '        saved = _S15_START.get(id(n))\n'
            '        if saved is None or saved[0] is not n:\n'
            '            raise U.Fail("처음 손실을 모릅니다. 학습하기 전에 같은 n 으로 L01-S15 채점 칸을 먼저 통과하거나, "\n'
            '                         "check(\\"L01-S15b\\", n, loss, start=처음 손실) 처럼 넘기세요.")\n'
            '        first = saved[1]\n'
            '    now = _model_loss(n, "n")\n'
            '    if not now < first:\n'
            '        if abs(now - first) <= ref()["S16"]["min_difference"]:\n'
            '            why = "손실이 그대로입니다. loss.backward() 뒤에 p.grad 가 채워지는지(L01-S11b), 갱신 줄이 있는지 보세요."\n'
            '        else:\n'
            '            why = ("갱신이 p.data += -0.05 * p.grad (빼기)인지, p.grad = 0.0 이 loss.backward() 앞에 있는지 보세요.")\n'
            '        raise U.Fail(f"손실이 줄지 않았습니다: 처음 {F(first)} → 지금 n 으로 다시 계산한 손실 {F(now)}. {why}")\n'
            '    r = ref()["S15b"]\n'
            '    limit = r["last_loss_max"]\n'
            '    if limit is not None and not now < limit:\n'
            '        raise U.Fail(f"지금 손실: 내 값 {F(now)}, 기준 {F(limit)} 미만. 학습 셀을 한 번 더 실행해 20번 더 학습한 뒤 다시 채점하세요.")\n'
            '    hint_max = r["teacher_value_range"][1]\n'
            '    note = ("" if now < hint_max else\n'
            '            f" 교사 안내: {F(hint_max)} 위면 학습 셀을 한 번 더 실행해 20번 더 학습해 보세요(채점은 줄었는지만 봅니다).")\n'
            '    return (f"손실이 줄었습니다: 처음 {F(first)} → 지금 {F(now)}.{note} 순전파, grad 비우기, 역전파, 갱신을 되풀이하는 것이 "\n'
            '            "학습의 전부입니다.")\n'
            '\n'
            '\n'
            '@grader("L01-S16", "grad 비우기를 빼먹는 버그", \'check("L01-S16", seen=True)\')\n'
            'def s16(losses_ok=None, losses_no_zero=None, seen=False):\n'
            '    """교사 정본은 눈으로 비교하는 단계라 check("L01-S16", seen=True) 로 표시한다.\n'
            '    손실 목록을 모았다면 check("L01-S16", losses_ok, losses_no_zero) 로 두 목록이 다른지 확인한다."""\n'
            '    if losses_ok is None and losses_no_zero is None and seen is True:\n'
            '        return ("눈으로 비교했다고 표시했습니다. grad 를 비우지 않으면 기울기가 반복마다 쌓여(+=) 손실이 고르게 줄지 않고 "\n'
            '                "튀곤 합니다. # 을 지워 되돌리고 다시 실행했는지 보세요.")\n'
            '    a = U.number_list(losses_ok, "첫 번째 손실 목록(grad를 비운 학습)")\n'
            '    b = U.number_list(losses_no_zero, "두 번째 손실 목록(grad를 비우지 않은 학습)")\n'
            '    if losses_ok is losses_no_zero:\n'
            '        raise U.Fail("두 칸에 같은 목록을 넘겼습니다. 두 번째는 p.grad = 0.0 줄을 막고 새로 돌린 손실 목록이어야 합니다.")\n'
            '    if not a[-1] < a[0]:\n'
            '        raise U.Fail(f"첫 번째 목록(grad를 비운 정상 학습)의 손실이 줄지 않았습니다: {F(a[0])} → {F(a[-1])}. "\n'
            '                     "S15b와 같은 학습인지 보세요.")\n'
            '    count = min(len(a), len(b))\n'
            '    diff = max(abs(x - y) for x, y in zip(a[:count], b[:count]))\n'
            '    if diff <= ref()["S16"]["min_difference"] and len(a) == len(b):\n'
            '        raise U.Fail("두 손실 목록이 똑같습니다. 두 번째는 p.grad = 0.0 줄을 막고 다시 돌린 결과여야 합니다.")\n'
            '    return "grad를 비우지 않으면 손실이 다르게 움직입니다. 기울기가 반복마다 쌓이기(+=) 때문에 매번 비워야 합니다."\n'
            '\n'
            '\n'
            '# ---- 재구성 3조각 -------------------------------------------------------------\n'
            '\n'
            '@grader("L01-R1", "Value에 +, *, tanh와 각각의 _backward", \'check("L01-R1", Value)\')\n'
            'def r1(Value=None):\n'
            '    """Value: 빈 노트북에서 다시 만든 __add__, __mul__, tanh 와 각각의 _backward (backward() 는 없어도 됨)."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-R1"])\n'
            '    t = tol()\n'
            '    a, b = U.make_value(Value, 2.0), U.make_value(Value, -3.0)\n'
            '    for desc, fn, want in (("a + b", lambda: a + b, -1.0), ("a * b", lambda: a * b, -6.0)):\n'
            '        out = U.student(f"{desc} 계산", fn)\n'
            '        U.require_value_like(out, f"{desc} 의 결과")\n'
            '        got = U.data_of(out, desc)\n'
            '        if not U.close(got, want, t):\n'
            '            raise U.Fail(f"{desc} (a = 2.0, b = -3.0): 내 값 {F(got)}, 맞는 값 {F(want)}.")\n'
            '    x = U.make_value(Value, 0.5)\n'
            '    o = U.student("Value(0.5).tanh() 계산", lambda: x.tanh())\n'
            '    U.require_value_like(o, "tanh() 의 결과")\n'
            '    got = U.data_of(o, "Value(0.5).tanh()")\n'
            '    if not U.close(got, math.tanh(0.5), t):\n'
            '        raise U.Fail(f"Value(0.5).tanh(): 내 값 {F(got)}, 맞는 값 {F(math.tanh(0.5))}.")\n'
            '    _check_add_mul_backward(Value, "")\n'
            '    _check_tanh_backward(Value, "")\n'
            '    # += 로 쌓기: _backward 를 두 번 부르면 두 배\n'
            '    a, b = U.make_value(Value, 2.0), U.make_value(Value, -3.0)\n'
            '    c = U.student("c = a*b 계산", lambda: a * b)\n'
            '    bw = U.get_attr(c, "_backward", "a*b 의 결과")\n'
            '    U.set_grad(c, 1.0, "c")\n'
            '    U.student("c._backward() 실행", bw)\n'
            '    U.student("c._backward() 한 번 더 실행", bw)\n'
            '    ga = U.grad_of(a, "a")\n'
            '    if not U.close(ga, -6.0, t):\n'
            '        raise U.Fail(f"c = a*b 에서 c._backward() 를 두 번 부른 뒤 a.grad: 내 값 {F(ga)}, 맞는 값 -6.0. "\n'
            '                     "_backward 안에서 = 대신 += 로 쌓아야 합니다.")\n'
            '    a = U.make_value(Value, 3.0)\n'
            '    s = U.student("b = a + a 계산", lambda: a + a)\n'
            '    U.require_value_like(s, "a + a 의 결과")\n'
            '    U.drive_backward(s)\n'
            '    ga = U.grad_of(a, "a")\n'
            '    if not U.close(ga, 2.0, t):\n'
            '        raise U.Fail(f"b = a + a 뒤 a.grad: 내 값 {F(ga)}, 맞는 값 2.0. _backward 안에서 = 대신 += 로 쌓아야 합니다.")\n'
            '    # 강의 뉴런 (채점기가 직접 거꾸로 돈다)\n'
            '    leaves, n = _lecture_neuron(Value)\n'
            '    o = U.student("o = n.tanh() 계산", lambda: n.tanh())\n'
            '    U.require_value_like(o, "n.tanh() 의 결과")\n'
            '    U.drive_backward(o)\n'
            '    mm = _neuron_mismatch(leaves, n, o)\n'
            '    if mm is not None:\n'
            '        raise U.Fail(_neuron_message(mm, "각 연산의 _backward 를 다시 보세요."))\n'
            '    done = U.compare_with_torch(Value, REUSE_EXPRS, False, t,\n'
            '                                hint="같은 값을 여러 번 쓰는 식입니다. _backward 안에서 += 로 쌓나요?")\n'
            '    return ("__add__, __mul__, tanh 와 각각의 _backward 가 원본 micrograd 와 같은 기울기를 냅니다."\n'
            '            + U.torch_note(done, len(REUSE_EXPRS)))\n'
            '\n'
            '\n'
            '@grader("L01-R2", "위상 정렬을 쓰는 backward()", \'check("L01-R2", Value)\')\n'
            'def r2(Value=None):\n'
            '    """Value: 빈 노트북에서 다시 만든 backward() (위상 정렬, self.grad = 1.0, reversed) 가 있는 클래스."""\n'
            '    U.need_callable(Value, "Value", USAGE["L01-R2"])\n'
            '    t = tol()\n'
            '    bw_hint = "Value에 backward() 메서드를 넣으세요."\n'
            '    leaves, n = _lecture_neuron(Value)\n'
            '    o = U.student("o = n.tanh() 계산", lambda: n.tanh())\n'
            '    U.require_value_like(o, "n.tanh() 의 결과")\n'
            '    U.student("o.backward() 실행", U.get_attr(o, "backward", "Value", hint=bw_hint))\n'
            '    ops_ref = "L01-R1(연산과 _backward)"\n'
            '    mm = _neuron_mismatch(leaves, n, o)\n'
            '    if mm is not None:\n'
            '        extra = "" if mm[0] == "o" else _why_backward_wrong(Value, REUSE_EXPRS, ops_ref)\n'
            '        raise U.Fail(_neuron_message(mm, extra))\n'
            '    r = ref()["R2"]["diamond"]\n'
            '\n'
            '    def diamond_grads(use_student_backward):\n'
            '        vals = {k: U.make_value(Value, r[k]) for k in ("a", "b", "c", "d")}\n'
            '\n'
            '        def build():\n'
            '            x = vals["a"] * vals["b"]\n'
            '            return (x + vals["c"]) + (x * vals["d"])\n'
            '\n'
            '        w = U.student("x = a*b; w = (x + c) + (x * d) 계산", build)\n'
            '        U.require_value_like(w, "w")\n'
            '        if use_student_backward:\n'
            '            U.student("w.backward() 실행", U.get_attr(w, "backward", "Value", hint=bw_hint))\n'
            '        else:\n'
            '            U.drive_backward(w)\n'
            '        return {k: U.grad_of(v, k) for k, v in vals.items()}\n'
            '\n'
            '    got = diamond_grads(True)\n'
            '    for name in ("a", "b", "c", "d"):\n'
            '        if not U.close(got[name], r["grads"][name], t):\n'
            '            ops = diamond_grads(False)\n'
            '            if all(U.close(ops[k], r["grads"][k], t) for k in ops):\n'
            '                why = "연산은 맞으니 backward() 쪽입니다. 위상 정렬에서 visited 로 같은 값을 한 번만 넣나요?"\n'
            '            else:\n'
            '                why = "같은 값을 두 길로 쓰면 기울기를 더해야 합니다. _backward 에서 += 로 쌓나요? (L01-R1)"\n'
            '            raise U.Fail(f"x = a*b 를 두 길로 쓰는 식 w = (x + c) + (x * d) 에서 {name}.grad: 내 값 {F(got[name])}, "\n'
            '                         f"맞는 값 {F(r[\'grads\'][name])}. {why}")\n'
            '    single = U.make_value(Value, 3.0)\n'
            '    U.student("Value(3.0).backward() 실행", U.get_attr(single, "backward", "Value", hint=bw_hint))\n'
            '    g = U.grad_of(single, "Value(3.0)")\n'
            '    if not U.close(g, 1.0, t):\n'
            '        raise U.Fail(f"재료 없는 Value(3.0).backward() 뒤 grad: 내 값 {F(g)}, 맞는 값 1.0. self.grad = 1.0 을 넣으세요.")\n'
            '    try:\n'
            '        done = U.compare_with_torch(Value, REUSE_EXPRS, True, t)\n'
            '    except U.Fail as exc:\n'
            '        raise U.Fail(f"{exc.message_ko} {_why_backward_wrong(Value, REUSE_EXPRS, ops_ref)}") from None\n'
            '    return "위상 정렬을 쓰는 backward() 가 원본 micrograd 와 같은 기울기를 냅니다." + U.torch_note(done, len(REUSE_EXPRS))\n'
            '\n'
            '\n'
            '@grader("L01-R3", "MLP(3, [4,4,1]) 학습으로 손실 0.1 아래", \'check("L01-R3", n, loss)\')\n'
            'def r3(n=None, loss=None):\n'
            '    """n: 빈 노트북에서 다시 만들어 학습한 MLP(3, [4, 4, 1]), loss: 마지막으로 계산한 손실(Value 또는 숫자)."""\n'
            '    last = _check_trained(n, loss, ref()["R3"]["last_loss_max"],\n'
            '                          "반복을 늘려(예: 100번 이상) 더 학습한 뒤 다시 채점하세요.")\n'
            '    return f"MLP(3, [4, 4, 1]) 학습 손실이 0.1 아래로 내려갔습니다(마지막 손실 {F(last)})."\n'
        ),
        'z2h_check.result_code': (
            '"""결과코드: Z2H1.{base64url(JSON)}.{crc32 8자리}\n'
            '\n'
            '규칙 (진도 서버·앱의 해독기와 같다. 바꾸지 않는다):\n'
            '- JSON은 키 순서 v, l, p, f, t 고정, 공백 없음.\n'
            '  예: {"v":1,"l":"L01","p":["L01-S11b"],"f":["L01-R2"],"t":"2026-11-16T21:00:00+09:00"}\n'
            '  p = 통과한 check_id, f = 실패한 check_id (L01-S11b 같은 전체 문자열). 둘 다 없으면 "아직 안 함".\n'
            '- base64url은 패딩(=)을 뗀다.\n'
            '- crc32는 base64url 문자열의 UTF-8 바이트로 계산해 8자리 소문자 16진수로 쓴다.\n'
            '비밀값이 아니며 위조를 막지 않는다. 목적은 붙여넣기 오타 잡기다.\n'
            '"""\n'
            'import base64\n'
            'import binascii\n'
            'import datetime\n'
            'import json\n'
            'import re\n'
            'import zlib\n'
            '\n'
            'PREFIX = "Z2H1"\n'
            'VERSION = 1\n'
            'KEYS = ("v", "l", "p", "f", "t")\n'
            'KST = datetime.timezone(datetime.timedelta(hours=9))\n'
            '\n'
            '_B64 = re.compile(r"[A-Za-z0-9_-]+")\n'
            '_CRC = re.compile(r"[0-9a-f]{8}")\n'
            '\n'
            'MSG_SHAPE = "결과코드 모양이 아닙니다. 노트북 finish() 아래에 나온 \'Z2H1.\'로 시작하는 줄 전체를 복사해 주세요."\n'
            'MSG_CHANGED = "코드가 잘렸거나 글자가 바뀌었습니다. 노트북에서 결과코드를 다시 복사해 주세요."\n'
            'MSG_CONTENT = "결과코드 안의 내용을 읽지 못했습니다. 노트북에서 finish()를 다시 실행해 새 코드를 받아 주세요."\n'
            'MSG_VERSION = "이 결과코드는 다른 버전에서 만들었습니다. 노트북의 준비 셀을 새로 받아 다시 실행해 주세요."\n'
            '\n'
            '\n'
            'class ResultCodeError(ValueError):\n'
            '    """결과코드를 풀지 못함. message_ko 는 화면에 그대로 보여 줄 한국어 안내."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'def now_kst_iso():\n'
            '    """지금 서울 시각, 예: 2026-11-16T21:00:00+09:00"""\n'
            '    return datetime.datetime.now(KST).replace(microsecond=0).isoformat()\n'
            '\n'
            '\n'
            'def crc32_hex(text):\n'
            '    return format(zlib.crc32(text.encode("utf-8")) & 0xFFFFFFFF, "08x")\n'
            '\n'
            '\n'
            'def encode_result_code(lecture_id, passed, failed, at=None):\n'
            '    """encode_result_code("L01", ["L01-S11b"], ["L01-R2"], "2026-11-16T21:00:00+09:00") -> "Z2H1.....32dccd52"\n'
            '\n'
            '    passed·failed 는 넘긴 순서 그대로 담는다. at 이 없으면 지금 서울 시각.\n'
            '    """\n'
            '    payload = {\n'
            '        "v": VERSION,\n'
            '        "l": str(lecture_id),\n'
            '        "p": [str(x) for x in passed],\n'
            '        "f": [str(x) for x in failed],\n'
            '        "t": at or now_kst_iso(),\n'
            '    }\n'
            '    text = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))\n'
            '    b64 = base64.urlsafe_b64encode(text.encode("utf-8")).decode("ascii").rstrip("=")\n'
            '    return f"{PREFIX}.{b64}.{crc32_hex(b64)}"\n'
            '\n'
            '\n'
            'def decode_result_code(code):\n'
            '    """결과코드를 풀어 {"v":1,"l":...,"p":[...],"f":[...],"t":...} 를 돌려준다.\n'
            '\n'
            '    앞뒤·중간의 공백과 줄바꿈은 무시한다. 틀리면 ResultCodeError(한국어 안내)를 던진다.\n'
            '    """\n'
            '    if not isinstance(code, str):\n'
            '        raise ResultCodeError(MSG_SHAPE)\n'
            '    text = "".join(code.split())\n'
            '    parts = text.split(".")\n'
            '    if len(parts) != 3 or parts[0] != PREFIX:\n'
            '        raise ResultCodeError(MSG_SHAPE)\n'
            '    _prefix, b64, crc = parts\n'
            '    if not _B64.fullmatch(b64) or not _CRC.fullmatch(crc):\n'
            '        raise ResultCodeError(MSG_CHANGED)\n'
            '    if crc32_hex(b64) != crc:\n'
            '        raise ResultCodeError(MSG_CHANGED)\n'
            '    try:\n'
            '        raw = base64.urlsafe_b64decode(b64 + "=" * (-len(b64) % 4))\n'
            '        obj = json.loads(raw.decode("utf-8"))\n'
            '    except (binascii.Error, ValueError, UnicodeDecodeError):\n'
            '        raise ResultCodeError(MSG_CONTENT) from None\n'
            '    if not isinstance(obj, dict) or set(obj) != set(KEYS):\n'
            '        raise ResultCodeError(MSG_CONTENT)\n'
            '    if type(obj["v"]) is not int or obj["v"] != VERSION:\n'
            '        raise ResultCodeError(MSG_VERSION)\n'
            '    if not isinstance(obj["l"], str) or not isinstance(obj["t"], str):\n'
            '        raise ResultCodeError(MSG_CONTENT)\n'
            '    for key in ("p", "f"):\n'
            '        if not isinstance(obj[key], list) or not all(isinstance(x, str) for x in obj[key]):\n'
            '            raise ResultCodeError(MSG_CONTENT)\n'
            '    return {key: obj[key] for key in KEYS}\n'
        ),
    }
    packages = {'z2h_check', 'z2h_check.graders'}
    data = {
        'reference/l01.json': (
            '{\n'
            '  "lecture_id": "L01",\n'
            '  "version": 1,\n'
            '  "_설명": "1편 채점 기준값과 허용 오차. 손으로 계산할 수 있는 값과 교사 정본(inputs/pedagogy/content/L01.json, c6d3e72)의 expected_check 값만 넣었다. 실행해야 아는 값은 null로 두고 옆에 \'<키>_주석\' 칸에 \'노트북 실행 후 확정\'을 적는다(노트북 작업자가 채움). 지금 null 칸은 없다. 이 파일을 고친 뒤에는 python -m z2h_check.bundle --write 로 묶음을 다시 만든다.",\n'
            '  "_출처": "강의 예제 숫자(S07 a=2, b=-3, c=10, f=-2 / S09 x1=2, x2=0, w1=-3, w2=1, b=6.8813735870195432)와 S15 데이터 xs, ys는 karpathy/nn-zero-to-hero lectures/micrograd 강의 노트북(MIT)에서 가져왔다. 범위 값은 교사 정본 L01.json expected_check.",\n'
            '  "tolerance": {\n'
            '    "grad": 1e-6,\n'
            '    "_grad_설명": "값과 기울기 비교: |학생 값 - 기준 값| <= grad * max(1, |기준 값|). karpathy/micrograd test/test_engine.py 의 1e-6과 같다.",\n'
            '    "float32": 1e-4,\n'
            '    "_float32_설명": "S13b에서 학생이 float32 텐서(.double() 없이)를 쓴 경우에만 쓴다."\n'
            '  },\n'
            '  "S01": {\n'
            '    "f_at": [[3.0, 20.0], [0.0, 5.0], [-1.0, 12.0], [2.0, 9.0]],\n'
            '    "_설명": "f(x) = 3x² - 4x + 5 손계산: f(3) = 27 - 12 + 5 = 20, f(0) = 5, f(-1) = 3 + 4 + 5 = 12, f(2) = 12 - 8 + 5 = 9"\n'
            '  },\n'
            '  "S02": {\n'
            '    "min": 13.99,\n'
            '    "max": 14.01,\n'
            '    "_설명": "교사 정본 expected_check. 손 계산: f\'(3) = 6*3 - 4 = 14, h를 쓰면 (f(3+h) - f(3))/h = 14 + 3h (h = 0.0001이면 14.0003). 이 범위는 h가 0.0033 이하일 때까지 받는다."\n'
            '  },\n'
            '  "S03": {\n'
            '    "a": {"min": -3.01, "max": -2.99},\n'
            '    "b": {"min": 1.99, "max": 2.01},\n'
            '    "c": {"min": 0.99, "max": 1.01},\n'
            '    "_설명": "d = a*b + c 는 a, b, c 각각에 대해 직선이라 숫자 기울기는 정확히 b(-3), a(2), 1 이다(반올림 오차만 있음)."\n'
            '  },\n'
            '  "S04": {"data": 2.0, "repr_prefix": "Value(data=2.0"},\n'
            '  "S05": {"a": 2.0, "b": -3.0, "c": 10.0, "ab": -6.0, "d": 4.0},\n'
            '  "S07": {\n'
            '    "_설명": "L = (a*b + c) * f, e = a*b, d = e + c. 기대 grad는 학생이 넘긴 값에서 연쇄 법칙으로 계산한다: L 1, f = d의 값, d = f의 값, e = c = d.grad, a = e.grad * b, b = e.grad * a",\n'
            '    "lecture_example": {\n'
            '      "a": 2.0, "b": -3.0, "c": 10.0, "f": -2.0,\n'
            '      "grads": {"L": 1.0, "f": 4.0, "d": -2.0, "e": -2.0, "c": -2.0, "a": 6.0, "b": -4.0}\n'
            '    }\n'
            '  },\n'
            '  "S08": {\n'
            '    "before": -8.0,\n'
            '    "after_min": -7.4,\n'
            '    "after_max": -7.2,\n'
            '    "_설명": "강의 숫자로 S07을 하면 L = -8.0. a = 2.06, b = -3.04, c = 9.98, f = -1.96 으로 움직이면 e = -6.2624, d = 3.7176, L = -7.286496 (교사 정본 범위 -7.4~-7.2). 전·후 두 값을 넘기면 커졌는지만 본다."\n'
            '  },\n'
            '  "S09": {"x1": 2.0, "x2": 0.0, "w1": -3.0, "w2": 1.0, "b": 6.8813735870195432, "n": 0.8813735870195432},\n'
            '  "S09b": {\n'
            '    "x_values": [-2.0, -0.5, 0.0, 0.8813735870195432, 1.5],\n'
            '    "o_lecture": 0.7071067811865476,\n'
            '    "_설명": "n = 0.8813735870195432 = ln(1 + √2) 이라 tanh(n) = 1/√2"\n'
            '  },\n'
            '  "S10": {"a": 2.0, "b": -3.0, "out_grads": [1.0, 2.0, -0.5]},\n'
            '  "S10b": {"x_values": [-2.0, -0.5, 0.0, 0.8813735870195432, 1.5], "out_grads": [1.0, 3.0]},\n'
            '  "S11b": {\n'
            '    "lecture_grads": {"o": 1.0, "n": 0.5, "x1": -1.5, "w1": 1.0, "x2": 0.5, "w2": 0.0, "b": 0.5},\n'
            '    "_설명": "o = tanh(n) = 1/√2 이라 1 - o² = 0.5. x1.grad = w1 * 0.5, w1.grad = x1 * 0.5, x2.grad = w2 * 0.5, w2.grad = x2 * 0.5, b.grad = 0.5"\n'
            '  },\n'
            '  "S12": {"a": 3.0, "a_plus_a": 2.0, "a_times_a": 6.0},\n'
            '  "S13": {\n'
            '    "cases": [\n'
            '      {"expr": "a + 1", "a": 2.0, "value": 3.0, "grad_a": 1.0},\n'
            '      {"expr": "1 + a", "a": 2.0, "value": 3.0, "grad_a": 1.0},\n'
            '      {"expr": "a * 2", "a": 3.0, "value": 6.0, "grad_a": 2.0},\n'
            '      {"expr": "2 * a", "a": 3.0, "value": 6.0, "grad_a": 2.0}\n'
            '    ],\n'
            '    "sum": {"values": [1.0, 2.0, 3.0], "value": 6.0},\n'
            '    "_설명": "교사 정본 확인 줄 print(Value(2.0) + 1, 1 + Value(2.0), 2 * Value(3.0)) -> 3.0 3.0 6.0. 기울기: a + 숫자는 1, 숫자 * a 는 그 숫자. sum() 은 0부터 더하므로 __radd__ 가 필요하다."\n'
            '  },\n'
            '  "S13b": {\n'
            '    "cases": [\n'
            '      {"expr": "a.exp()", "a": 0.0, "value": 1.0, "grad_a": 1.0},\n'
            '      {"expr": "a.exp()", "a": 1.0, "value": 2.718281828459045, "grad_a": 2.718281828459045},\n'
            '      {"expr": "-a", "a": 5.0, "value": -5.0, "grad_a": -1.0},\n'
            '      {"expr": "a - b", "a": 5.0, "b": 2.0, "value": 3.0, "grad_a": 1.0, "grad_b": -1.0},\n'
            '      {"expr": "a - 1", "a": 3.0, "value": 2.0, "grad_a": 1.0}\n'
            '    ],\n'
            '    "_설명": "exp(x)의 기울기는 exp(x) 자신(e = 2.718281828459045), -a 는 -1, a - b 는 (1, -1). 교사 정본 확인 줄의 Value(3.0) - 1 = 2.0, Value(0.0).exp() = 1.0 포함."\n'
            '  },\n'
            '  "S13c": {\n'
            '    "cases": [\n'
            '      {"expr": "a ** 2", "a": 2.0, "value": 4.0, "grad_a": 4.0},\n'
            '      {"expr": "a ** -1", "a": 2.0, "value": 0.5, "grad_a": -0.25},\n'
            '      {"expr": "a ** 0.5", "a": 4.0, "value": 2.0, "grad_a": 0.25},\n'
            '      {"expr": "a / 2", "a": 6.0, "value": 3.0, "grad_a": 0.5},\n'
            '      {"expr": "a / b", "a": 5.0, "b": 2.0, "value": 2.5, "grad_a": 0.5, "grad_b": -1.25}\n'
            '    ],\n'
            '    "_설명": "a**k 의 기울기는 k * a**(k-1): 2*2 = 4, -1 * 2**-2 = -0.25, 0.5 * 4**-0.5 = 0.25. a / b 는 (1/b, -a/b²) = (0.5, -1.25). 교사 정본 확인 줄의 Value(2.0)**2 = 4.0, Value(6.0) / 2 = 3.0 포함."\n'
            '  },\n'
            '  "S14": {"cases": [{"nin": 2, "x": [2.0, 3.0]}, {"nin": 3, "x": [1.0, -2.0, 0.5]}], "repeat": 3},\n'
            '  "S14b": {"cases": [{"nin": 2, "nout": 3, "x": [2.0, 3.0]}, {"nin": 3, "nout": 2, "x": [1.0, -2.0, 0.5]}]},\n'
            '  "S14c": {\n'
            '    "cases": [{"nin": 3, "nouts": [4, 4, 1], "count": 41}, {"nin": 2, "nouts": [3, 1], "count": 13}],\n'
            '    "x": [2.0, 3.0, -1.0],\n'
            '    "_설명": "층마다 (입력 수 + 1) × 출력 수를 더한다: 3→4 16, 4→4 20, 4→1 5, 합 41 / 2→3 9, 3→1 4, 합 13"\n'
            '  },\n'
            '  "S15": {\n'
            '    "xs": [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]],\n'
            '    "ys": [1.0, -1.0, -1.0, 1.0],\n'
            '    "params": 41,\n'
            '    "loss_upper_bound": 16.0,\n'
            '    "_loss_upper_bound_설명": "정답이 ±1이고 tanh 출력은 -1~1이라 오차 제곱 하나는 4보다 작다. 데이터 4개면 합은 16보다 작다(교사 정본: 학습 전 손실은 보통 1에서 8 사이).",\n'
            '    "_params_설명": "교사 정본 S15 expected_check len(n.parameters()) = 41"\n'
            '  },\n'
            '  "S15b": {\n'
            '    "last_loss_max": null,\n'
            '    "last_loss_max_주석": "노트북 실행 후 확정: 교사 정본(origin/m0-setup L01.json)에 시드 고정이 없어 실행마다 마지막 손실이 다르다(정답 구현 100번: 중앙 0.054, 95% 0.26, 최대 2.12). null 이면 S15 시작 손실보다 줄었는지만 본다(PM 결정).",\n'
            '    "teacher_value_range": [0.0, 0.1],\n'
            '    "_teacher_value_range_설명": "교사 정본 S15b expected_check(마지막 줄 손실 0.0~0.1). 채점 기준이 아니라 통과 문구의 안내(0.1 위면 셀을 한 번 더 실행)에만 쓴다."\n'
            '  },\n'
            '  "S16": {"min_difference": 1e-9},\n'
            '  "R2": {\n'
            '    "diamond": {"a": 2.0, "b": -3.0, "c": 10.0, "d": -2.0, "grads": {"a": 3.0, "b": -2.0, "c": 1.0, "d": -6.0}},\n'
            '    "_설명": "x = a*b 를 두 길로 쓴다: w = (x + c) + (x * d). dw/da = b(1 + d) = 3, dw/db = a(1 + d) = -2, dw/dc = 1, dw/dd = a*b = -6"\n'
            '  },\n'
            '  "R3": {\n'
            '    "last_loss_max": 0.1,\n'
            '    "params": 41,\n'
            '    "_설명": "교사 정본 rebuild_tasks L01-R3: MLP(3, [4, 4, 1])을 S15 데이터로 학습해 마지막 손실 0.0~0.1"\n'
            '  }\n'
            '}\n'
        ),
    }

    class _Z2HBundleFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        z2h_bundle = True

        def find_spec(self, fullname, path=None, target=None):
            if fullname in sources:
                return importlib.util.spec_from_loader(fullname, self, origin="<z2h_check 묶음>",
                                                       is_package=fullname in packages)
            return None

        def create_module(self, spec):
            return None

        def exec_module(self, module):
            name = module.__name__
            tail = "/__init__.py" if name in packages else ".py"
            filename = "<z2h_check 묶음>/" + name.replace(".", "/") + tail
            module.__file__ = filename
            if name == "z2h_check":
                module.__z2h_bundled__ = True
            exec(compile(sources[name], filename, "exec"), module.__dict__)
            if name == "z2h_check._resources":
                module._EMBEDDED.update(data)

    for finder in list(sys.meta_path):
        if getattr(finder, "z2h_bundle", False):
            sys.meta_path.remove(finder)
    for name in list(sys.modules):
        if name == "z2h_check" or name.startswith("z2h_check."):
            del sys.modules[name]
    sys.meta_path.insert(0, _Z2HBundleFinder())
    import z2h_check
    return z2h_check


if __name__ != "z2h_check._bundle":
    _z2h = _z2h_install()
    check, finish = _z2h.check, _z2h.finish
    print("z2h_check 준비 완료 (묶음 0.1.0): check(...)로 채점하고, 마지막에 finish(\"L01\")를 실행하세요.")
    del _z2h


# ---- 여기부터는 이 노트북의 채점 셀 도우미입니다(z2h_check 묶음 밖, build_notebooks.py 가 붙임) ----
___ = None  # 빈칸(___)을 쓰는 노트북을 위한 자리(z2h_check/README.md)


class _Z2HNotShown(NameError):
    """바로 위 셀이 아직 값을 보여 주지 않았을 때."""


def _z2h_shown():
    """바로 앞에서 셀의 마지막 줄이 화면에 보여 준 값(IPython 의 Out 기록 중 가장 최근 것)을 돌려줍니다.
    위의 ___ = None 때문에 IPython 이 _ 를 더 이상 바꾸지 않으므로 _ 대신 Out 을 씁니다."""
    shown = globals().get("Out")
    if not shown:
        raise _Z2HNotShown("바로 위 셀이 보여 준 값이 아직 없습니다")
    return shown[max(shown)]


def _z2h_not_yet(check_id, err):
    """채점 셀에서 check(...)에 넘길 값이 아직 없거나 만들다 오류가 나면, 예외 대신 "○ 아직" 한 줄만 보여 줍니다.
    이때는 채점 기록을 남기지 않으므로 finish("L01") 채점표에는 "아직 안 함"으로 나옵니다."""
    try:
        if isinstance(err, _Z2HNotShown):
            print(f"○ 아직: 바로 위 단계 셀을 먼저 실행한 뒤 이 채점 셀을 다시 실행하세요. [{check_id}]")
        elif isinstance(err, NameError):
            name = getattr(err, "name", None)
            what = f"이름 {name}" if name else "이름"
            print(f"○ 아직: 이 칸에 필요한 {what}이(가) 아직 없어요. 위 단계 셀을 먼저 실행한 뒤 "
                  f"이 채점 셀을 다시 실행하세요. [{check_id}]")
        else:
            text = " ".join(str(err).split())[:120]
            print(f"○ 아직: 채점에 넘길 값을 만들다 오류가 났어요({type(err).__name__}: {text}). "
                  f"위 단계 셀이 오류 없이 실행됐는지 보고 이 채점 셀을 다시 실행하세요. [{check_id}]")
    except Exception:
        print(f"○ 아직: 위 단계 셀을 먼저 실행하세요. [{check_id}]")


## L01-S01 함수 그래프 그리기

영상: 0:08:08 입력 1개 함수의 미분 ([0:08:08부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=488&end=852&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 아래 9줄을 입력하고 실행합니다. 앞 4줄은 이 편에서 쓸 도구를 불러오는 줄입니다(F-P8).

**보여야 하는 것**: 아래로 볼록한 포물선(U 모양) 그래프가 보입니다. 가장 낮은 곳은 x가 약 0.67인 부근입니다. 그래프 위에 `[<matplotlib.lines.Line2D ...>]` 같은 글이 한 줄 보여도 정상입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S01", f)
except Exception as _err:
    _z2h_not_yet("L01-S01", _err)

## L01-S02 숫자 기울기

영상: 0:08:08 입력 1개 함수의 미분 ([0:08:08부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=488&end=852&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 3줄을 입력하고 실행합니다.

**보여야 하는 것**: 약 14.0003 이 보입니다(14에 아주 가까운 수).

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S02", _z2h_shown())   # 바로 위 셀의 마지막 줄이 보여 준 값(숫자 기울기)
except Exception as _err:
    _z2h_not_yet("L01-S02", _err)

## L01-S03 입력이 여러 개일 때

영상: 0:14:12 입력 여러 개 함수의 미분 ([0:14:12부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=852&end=1149&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 7줄을 입력하고 실행합니다. a만 조금 올려서 d가 얼마나 변하는지 봅니다.

**보여야 하는 것**: `4.0 3.999699999999999 -3.000000000010772`처럼 세 숫자가 보입니다. 마지막 숫자는 약 -3.0(b의 값)입니다. 소수 끝자리가 지저분한 것은 컴퓨터 소수 계산의 작은 오차라 정상입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S03", (d2 - d1) / h)   # 위 셀이 만든 d1, d2, h 로 a에 대한 기울기를 다시 계산해 넘깁니다
except Exception as _err:
    _z2h_not_yet("L01-S03", _err)

## L01-S04 Value 클래스 뼈대

영상: 0:19:09 Value 객체, draw_dot 시각화 ([0:19:09부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 Value 클래스 8줄과 확인 1줄을 입력하고 실행합니다. 이 셀을 앞으로 'Value 클래스 셀'이라 부르며, S05부터 이 셀 안에 줄을 더해 갑니다.

**보여야 하는 것**: `Value(data=2.0)`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S04", Value)
except Exception as _err:
    _z2h_not_yet("L01-S04", _err)

## L01-S05 덧셈과 곱셈 기록하기

영상: 0:19:09 Value 객체, draw_dot 시각화 ([0:19:09부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀의 맨 아래(`__repr__` 다음)에 6줄을 더하고 그 셀을 다시 실행합니다. 그다음 새 셀에 확인 1줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 6줄 넣기

**실행 순서**: 고친 L01-S04 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: `Value(data=4.0)`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S05", Value)
except Exception as _err:
    _z2h_not_yet("L01-S05", _err)

## L01-S06 계산 그래프 그림

영상: 0:19:09 Value 객체, draw_dot 시각화 ([0:19:09부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 준비 코드 셀에 그림 함수 draw_dot을 붙여 넣고 실행합니다(직접 치지 않음). 그다음 새 셀에 6줄을 입력하고 실행합니다.

**준비 코드**(직접 치지 않고 아래 준비 코드 셀을 실행만 합니다): karpathy/nn-zero-to-hero lectures/micrograd 노트북의 draw_dot를 줄여 다듬은 것(label 칸 대신 data와 grad만 표시)

**실행 순서**: 준비 코드 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: a, b, c, e, d가 상자로, `*`와 `+`가 동그라미로 이어진 그림이 보입니다. 모든 상자의 grad는 아직 0.0000입니다.

In [ ]:
from graphviz import Digraph
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges
def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label=f"data {n.data:.4f} | grad {n.grad:.4f}", shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S06", draw_dot(d))   # graphviz를 못 쓰는 곳이면 이 줄을 check("L01-S06", seen=True) 로 바꿉니다
except Exception as _err:
    _z2h_not_yet("L01-S06", _err)

## L01-S07 손으로 기울기 채우기

영상: 0:32:10 손으로 역전파 예제 1 ([0:32:10부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 10줄을 입력하고 실행합니다. 결과 L에서 시작해 거꾸로 기울기를 직접 채웁니다.

**보여야 하는 것**: L의 data는 -8.0000입니다. grad: L 1.0, d -2.0, f 4.0, c -2.0, e -2.0, a 6.0, b -4.0.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S07", a=a, b=b, c=c, d=d, e=e, f=f, L=L)
except Exception as _err:
    _z2h_not_yet("L01-S07", _err)

## L01-S08 한 걸음 움직여 보기

영상: 0:51:10 최적화 한 단계 미리보기 ([0:51:10부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3070&end=3172&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 6줄을 입력하고 실행합니다. 기울기 방향으로 값을 조금 움직이고 L을 다시 계산합니다.

**보여야 하는 것**: 약 -7.29가 보입니다. 처음 -8.0보다 커졌습니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S08", L)
except Exception as _err:
    _z2h_not_yet("L01-S08", _err)

## L01-S09 뉴런 하나 계산하기

영상: 0:52:52 손으로 역전파 예제 2: 뉴런 ([0:52:52부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 7줄을 입력하고 실행합니다. 입력 2개, 가중치 2개, 편향 1개로 뉴런의 합을 계산합니다.

**보여야 하는 것**: `Value(data=0.8813735870195432)`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S09", n, x1=x1, x2=x2, w1=w1, w2=w2, b=b)
except Exception as _err:
    _z2h_not_yet("L01-S09", _err)

## L01-S09b tanh 붙이기

영상: 0:52:52 손으로 역전파 예제 2: 뉴런 ([0:52:52부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 맨 아래에 tanh 4줄을 더하고 다시 실행합니다. 클래스를 바꿨으니 S09 셀도 다시 실행한 뒤, 새 셀에 2줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 4줄 넣기

**먼저 다시 실행할 칸**: L01-S09 칸의 셀

**실행 순서**: 고친 L01-S04 셀 → L01-S09 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: 그림 맨 오른쪽 o 상자의 data가 0.7071입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S09b", Value, o=o)
except Exception as _err:
    _z2h_not_yet("L01-S09b", _err)

## L01-S10 _backward를 + 와 * 안에 넣기

영상: 1:09:02 연산별 _backward ([1:09:02부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4142&end=4652&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 세 곳에 줄을 더하고 다시 실행합니다. 이 단계는 실행 결과가 따로 없고, 확인은 S10b에서 합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `__init__` 안, `self.grad = 0.0` 바로 아래 · 기준 줄: `self.grad = 0.0` · 1줄 넣기
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `__add__` 안, `out = ...` 줄과 `return out` 사이 · 기준 줄: `out = Value(self.data + other.data, (self, other), '+')` · 4줄 넣기
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `__mul__` 안, `out = ...` 줄과 `return out` 사이 · 기준 줄: `out = Value(self.data * other.data, (self, other), '*')` · 4줄 넣기

**실행 순서**: 고친 L01-S04 셀 → 채점 셀

**보여야 하는 것**: 클래스 셀이 에러 없이 실행됩니다. 화면에 따로 보이는 것은 없습니다.

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S10", Value)
except Exception as _err:
    _z2h_not_yet("L01-S10", _err)

## L01-S10b tanh의 _backward

영상: 1:09:02 연산별 _backward ([1:09:02부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4142&end=4652&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: tanh 안에 3줄을 더하고 클래스 셀을 다시 실행합니다. S09, S09b 셀을 다시 실행한 뒤 새 셀에 4줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `tanh` 안, `out = ...` 줄과 `return out` 사이 · 기준 줄: `out = Value(t, (self,), 'tanh')` · 3줄 넣기

**먼저 다시 실행할 칸**: L01-S09 칸의 셀, L01-S09b 칸의 셀

**실행 순서**: 고친 L01-S04 셀 → L01-S09 셀 → L01-S09b 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: n의 grad가 0.5000, b의 grad가 0.5000입니다. 그보다 왼쪽 상자들은 아직 0입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S10b", Value)
except Exception as _err:
    _z2h_not_yet("L01-S10b", _err)

## L01-S11 위상 정렬

영상: 1:17:32 그래프 전체 backward() ([1:17:32부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4652&end=4948&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 10줄을 입력하고 실행합니다. 상자들을 '재료가 먼저, 결과가 나중' 순서로 줄 세웁니다.

**보여야 하는 것**: Value 10개가 담긴 목록이 보입니다. 맨 뒤가 o(`Value(data=0.7071...)`)입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S11", topo, o)
except Exception as _err:
    _z2h_not_yet("L01-S11", _err)

## L01-S11b backward() 메서드

영상: 1:17:32 그래프 전체 backward() ([1:17:32부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4652&end=4948&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 맨 아래에 backward를 더합니다. 13줄 가운데 8줄은 S11에서 쓴 줄을 그대로 옮겨 붙이는 것이고, 새로 생각할 줄은 5줄입니다. 클래스 셀, S09, S09b를 다시 실행한 뒤 새 셀에 2줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 13줄 넣기

**먼저 다시 실행할 칸**: L01-S09 칸의 셀, L01-S09b 칸의 셀

**실행 순서**: 고친 L01-S04 셀 → L01-S09 셀 → L01-S09b 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: grad: o 1.0, n 0.5, b 0.5, x1 -1.5, w1 1.0, x2 0.5, w2 0.0.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S11b", Value)
except Exception as _err:
    _z2h_not_yet("L01-S11b", _err)

## L01-S12 같은 값을 두 번 쓰는 버그

영상: 1:22:28 여러 번 쓰는 노드 버그 ([1:22:28부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4948&end=5225&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 4줄을 입력하고 실행합니다.

**보여야 하는 것**: `2.0`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S12", Value)
except Exception as _err:
    _z2h_not_yet("L01-S12", _err)

## L01-S13 숫자와 섞어 계산하기

영상: 1:27:05 tanh 쪼개기(더 많은 연산) ([1:27:05부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 네 곳에 줄을 더하고 다시 실행한 뒤, 새 셀에 확인 1줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `__add__`의 첫 줄(`def __add__...`) 바로 아래 · 기준 줄: `def __add__(self, other):` · 1줄 넣기
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: `__mul__`의 첫 줄 바로 아래 · 기준 줄: `def __mul__(self, other):` · 1줄 넣기
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 4줄 넣기

**실행 순서**: 고친 L01-S04 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: `Value(data=3.0) Value(data=3.0) Value(data=6.0)`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S13", Value)
except Exception as _err:
    _z2h_not_yet("L01-S13", _err)

## L01-S13b exp와 빼기

영상: 1:27:05 tanh 쪼개기(더 많은 연산) ([1:27:05부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 맨 아래에 10줄을 더하고 다시 실행합니다. 확인은 S13c 마지막 줄에서 함께 합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 10줄 넣기

**실행 순서**: 고친 L01-S04 셀 → 채점 셀

**보여야 하는 것**: 클래스 셀이 에러 없이 실행됩니다. 화면에 따로 보이는 것은 없습니다.

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S13b", Value)
except Exception as _err:
    _z2h_not_yet("L01-S13b", _err)

## L01-S13c 거듭제곱과 나누기

영상: 1:27:05 tanh 쪼개기(더 많은 연산) ([1:27:05부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: Value 클래스 셀 맨 아래에 8줄을 더하고 다시 실행한 뒤, 새 셀에 확인 1줄을 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S04 칸에 쓴 셀 · 넣을 곳: Value 클래스 셀 맨 아래 · 8줄 넣기

**실행 순서**: 고친 L01-S04 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: `Value(data=2.0) Value(data=4.0) Value(data=3.0) Value(data=1.0)`이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S13c", Value)
except Exception as _err:
    _z2h_not_yet("L01-S13c", _err)

## L01-S13d tanh를 쪼개서 다시 계산하기

영상: 1:27:05 tanh 쪼개기(더 많은 연산) ([1:27:05부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: S09 셀을 다시 실행한 뒤 새 셀에 4줄을 실행합니다. tanh를 exp, 빼기, 나누기로 풀어 써서 같은 답이 나오는지 봅니다.

**먼저 다시 실행할 칸**: L01-S09 칸의 셀

**실행 순서**: L01-S09 셀 → 아래 빈 셀 → 채점 셀

**보여야 하는 것**: o의 data는 0.7071, x1 grad -1.5, w1 grad 1.0, x2 grad 0.5, w2 grad 0.0으로 S11b와 같습니다. 그림은 상자가 더 많습니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S13d", Value)
except Exception as _err:
    _z2h_not_yet("L01-S13d", _err)

## L01-S13e PyTorch로 비교

영상: 1:39:31 PyTorch 비교 ([1:39:31부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5971&end=6235&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 9줄을 입력하고 실행합니다. 같은 뉴런을 PyTorch로 계산해 기울기를 비교합니다.

**보여야 하는 것**: `0.7071067811865476 -1.4999999999999998 0.9999999999999999 0.49999999999999994 0.0`처럼, S11b와 같은 숫자(0.7071, -1.5, 1.0, 0.5, 0.0)에 아주 가까운 값이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S13e", x1=x1, x2=x2, w1=w1, w2=w2, b=b, o=o)   # torch가 없으면 "건너뜀"으로 나옵니다
except Exception as _err:
    _z2h_not_yet("L01-S13e", _err)

## L01-S14 Neuron 클래스

영상: 1:43:55 MLP 라이브러리 ([1:43:55부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 Neuron 클래스 9줄과 확인 1줄을 입력하고 실행합니다.

**보여야 하는 것**: `Value(data=...)`가 보이고, 숫자는 -1과 1 사이입니다(무작위라 실행마다 다름).

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S14", Neuron)
except Exception as _err:
    _z2h_not_yet("L01-S14", _err)

## L01-S14b Layer 클래스

영상: 1:43:55 MLP 라이브러리 ([1:43:55부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 Layer 클래스 8줄과 확인 1줄을 입력하고 실행합니다.

**보여야 하는 것**: Value 3개가 담긴 목록이 보입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S14b", Layer)
except Exception as _err:
    _z2h_not_yet("L01-S14b", _err)

## L01-S14c MLP 클래스와 parameters()

영상: 1:43:55 MLP 라이브러리 ([1:43:55부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 MLP 클래스 10줄을 입력하고 실행합니다(이 클래스는 한 덩어리라 나누지 않습니다).

**보여야 하는 것**: 셀이 에러 없이 실행됩니다. 화면에 따로 보이는 것은 없습니다. 파라미터 개수(41개)는 S15 첫 출력에서 확인합니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S14c", MLP)
except Exception as _err:
    _z2h_not_yet("L01-S14c", _err)

## L01-S15 데이터와 손실

영상: 1:51:04 데이터와 손실, 1:57:56 파라미터 모으기 ([1:51:04부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6664&end=7076&cc_load_policy=1&cc_lang_pref=ko&hl=ko) · [1:57:56부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7076&end=7272&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 7줄을 입력하고 실행합니다. 신경망의 파라미터 개수를 세고, 작은 데이터 4개와, 얼마나 틀렸는지 재는 손실을 만듭니다.

**보여야 하는 것**: 첫 줄에 `41`, 둘째 줄에 `Value(data=...)`가 보입니다. 손실 숫자는 실행마다 다르고 보통 1에서 8 사이입니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S15", n, loss)   # 학습(S15b) 전에 채점합니다
except Exception as _err:
    _z2h_not_yet("L01-S15", _err)

## L01-S15b 학습 반복

영상: 2:01:12 경사 하강 학습 ([2:01:12부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: 새 셀에 9줄을 입력하고 실행합니다. 순전파, 기울기 비우기, 역전파, 갱신을 20번 반복합니다.

**보여야 하는 것**: 20줄이 보이고 손실이 대체로 줄어듭니다. 마지막 손실은 대개 0.1 아래입니다(시작값이 무작위라 실행마다 다름). 0.1 위면 셀을 한 번 더 실행해 20번 더 학습합니다.

In [ ]:
# 여기에 작성

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    check("L01-S15b", n, loss)
except Exception as _err:
    _z2h_not_yet("L01-S15b", _err)

## L01-S16 grad 비우기를 빼먹는 버그

영상: 2:01:12 경사 하강 학습 ([2:01:12부터 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko))

**입력할 것**: S15b 셀에서 grad 비우기 2줄 앞에 `#`을 붙여 막고, S15 셀과 S15b 셀을 다시 실행해 손실을 비교합니다. 비교한 뒤 `#`을 지워 되돌립니다. 새로 치는 코드는 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다)
- 고칠 셀: L01-S15b 칸에 쓴 셀 · 넣을 곳: L01-S15b 셀의 grad 비우기 2줄 · 2줄 바꾸기

**먼저 다시 실행할 칸**: L01-S15 칸의 셀

**실행 순서**: L01-S15 셀 → 고친 L01-S15b 셀 → 채점 셀

**보여야 하는 것**: 손실이 고르게 줄지 않고 중간에 튀어 오르는 모습이 자주 보입니다(실행마다 다름). 이 작은 문제에서는 튄 뒤 오히려 더 빨리 0에 가까워지는 것처럼 보일 수도 있습니다. 쌓인 기울기가 우연히 큰 걸음이 된 것일 뿐이고, 큰 문제에서는 학습을 망칩니다. 되돌리고 다시 실행하면 S15b처럼 고르게 줄어듭니다.

In [ ]:
# 채점 셀: 고치지 말고 실행만 하세요.
try:
    n, loss   # 학습 셀(L01-S15, L01-S15b)이 만든 n과 loss가 있을 때만 아래 표시를 합니다
    check("L01-S16", seen=True)   # 눈으로 비교하는 단계라 "봤어요" 표시
except Exception as _err:
    _z2h_not_yet("L01-S16", _err)

## 마무리: 채점표와 결과코드

아래 셀을 실행하면 채점표와 결과코드가 나옵니다. 맨 아래 줄의 결과코드를 앱의 '결과코드 붙여넣기' 칸에 붙여 넣으세요. 로컬 Jupyter에서 진도 서버가 켜져 있으면 결과를 서버로도 보내 보고, Colab에서는 "서버에 닿지 못했습니다"가 정상입니다.

채점표의 L01-R1~R3(재구성 과제)는 이 노트북에서 채점하지 않으므로 "아직 안 함"으로 보입니다.

In [ ]:
# 마지막 칸: 채점표와 결과코드를 보여 줍니다. 맨 아래 줄(Z2H1. 로 시작)을 앱의 '결과코드 붙여넣기' 칸에 넣으세요.
result_code = finish("L01")